In [ ]:
!pip install rapidfuzz
import requests
import json
import random
import time
import itertools
from rapidfuzz import fuzz
from google.colab import userdata
import re
import os
import hashlib
API_KEY = userdata.get('PoTKey')

In [ ]:
def makeQuestionId(contextId, qKey):
    return f"{contextId}__{qKey}"

def classifyAnswer(parsedAnswer, correctAnswer):
    """
    parsedAnswer:
        valid answer -> A/B/C/D
        None         -> formatting/model-output failure
    """
    if parsedAnswer is None:
        return "invalid", 0
    if parsedAnswer == correctAnswer:
        return "correct", 1
    return "wrong", 0

def makeQuestionRecord(
    contextId,
    qKey,
    qNum,
    dagType,
    model,
    condition,
    correctAnswer,
    modelAnswer,
    rawOutput,
    status,
    isCorrect,
    error=None
):
    return {
        "question_id": makeQuestionId(contextId, qKey),
        "context_id": contextId,
        "q_key": qKey,
        "q_num": qNum,
        "topology": dagType,
        "model": model,
        "condition": condition,
        "correct_answer": correctAnswer,
        "model_answer": modelAnswer,
        "raw_output": rawOutput,
        "status": status,
        "correct": isCorrect,
        "error": error
    }

def saveQuestionResults(questionResults, outputPath):
    """
    Atomic save so interrupted writes don't destroy the file.
    """
    tempPath = outputPath + ".tmp"
    with open(tempPath, "w") as f:
        json.dump(questionResults, f, indent=2)
    os.replace(tempPath, outputPath)

def saveFailedQuestions(questionResults, outputPath):
    """
    Separate convenience file containing anything needing inspection/rerun.
    """
    failed = [
        row for row in questionResults
        if row["status"] in {"api_error", "invalid"}
    ]
    tempPath = outputPath + ".tmp"
    with open(tempPath, "w") as f:
        json.dump(failed, f, indent=2)
    os.replace(tempPath, outputPath)

In [ ]:
models = ['openai/gpt-4o',
          'openai/gpt-4o-mini',
          'anthropic/claude-sonnet-4.5',
          'meta-llama/llama-3.3-70b-instruct',
          'meta-llama/llama-3.1-8b-instruct']
with open('/content/causal_reasoning_dataset_shuffled_v1.json', 'r') as f:
  dataset = json.load(f)
with open('/content/handmade_true_dags.json', 'r') as f:
  trueDAGs = json.load(f)

In [ ]:
class ContentRefusalError(Exception):
    """Raised when the provider refuses the prompt for content-policy reasons."""
    pass
PROVIDER_MAP = {
    "openai/gpt-4o": "OpenAI",
    "openai/gpt-4o-mini": "OpenAI",
    "anthropic/claude-sonnet-4.5": "Amazon Bedrock",
    "meta-llama/llama-3.3-70b-instruct": "DeepInfra",
    "meta-llama/llama-3.1-8b-instruct": "DeepInfra"
}

def prompt(message, model="openai/gpt-4o", numtoken=100):
    for attempt in range(1, 11):
        try:
            request_json = {
                "model": model,
                "messages": [
                    {
                        "role": "user",
                        "content": message
                    }
                ],
                "max_tokens": numtoken,
                "temperature": 0
            }
            provider = PROVIDER_MAP.get(model)
            if provider is not None:
                request_json["provider"] = {
                    "only": [provider],
                    "allow_fallbacks": False
                }
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {API_KEY}",
                    "Content-Type": "application/json"
                },
                json=request_json,
                timeout=90
            )
            result = response.json()
            if "choices" not in result:
                print(
                    f"API Error (attempt {attempt}) | "
                    f"model={model} | "
                    f"requested_provider={provider} | "
                    f"response={result}"
                )
                time.sleep(2 * attempt)
                continue
            choice = result["choices"][0]
            message_obj = choice["message"]
            content = (
                message_obj.get("content")
                or message_obj.get("reasoning")
                or ""
            )
            finish_reason = choice.get("finish_reason")
            native_finish_reason = choice.get("native_finish_reason")
            if (
                finish_reason == "content_filter"
                or native_finish_reason == "refusal"
            ):
                print(
                    f"CONTENT REFUSAL -- skipping immediately | "
                    f"model={model} | "
                    f"requested_provider={provider} | "
                    f"actual_provider={result.get('provider')} | "
                    f"finish_reason={finish_reason} | "
                    f"native_finish_reason={native_finish_reason}"
                )
                raise ContentRefusalError(
                    f"Content refusal | model={model} | provider={provider}"
                )
            if not content.strip():
                print(
                    f"Empty completion (attempt {attempt}) | "
                    f"model={model} | "
                    f"requested_provider={provider} | "
                    f"actual_provider={result.get('provider')} | "
                    f"finish_reason={finish_reason} | "
                    f"native_finish_reason={native_finish_reason}"
                )
                time.sleep(2 * attempt)
                continue
            return content.strip()
        except ContentRefusalError:
            raise
        except Exception as e:
            print(
                f"Exception (attempt {attempt}) | "
                f"model={model} | "
                f"provider={PROVIDER_MAP.get(model)} | "
                f"{e}"
            )
            time.sleep(2 * attempt)
    raise RuntimeError(
        f"API call failed after 10 attempts | "
        f"model={model} | "
        f"provider={PROVIDER_MAP.get(model)}"
    )

In [ ]:
class fullGraph:
    def askCausalQ(self, model, context, tuplePair, pairs, promptMediator=False):
        if promptMediator == False:
            message1 = f"""
            You will be given a scenario and two variables from it, Variable A and Variable B. Your job is to judge whether Variable A causally influences Variable B.

            Scenario: {context}
            Variable A: {tuplePair[0]}
            Variable B: {tuplePair[1]}

            If Variable A ({tuplePair[0]}) were changed or removed, would Variable B ({tuplePair[1]}) change in any way, based on this scenario?
            Output ONLY yes or no with no explanation, and no punctuation, all lowercase.
            """
            message2 = f"""
            You will be given a scenario and two variables from it, Variable A and Variable B. Your job is to judge whether Variable A causally influences Variable B.

            Scenario: {context}
            Variable A: {tuplePair[1]}
            Variable B: {tuplePair[0]}

            If Variable A ({tuplePair[1]}) were changed or removed, would Variable B ({tuplePair[0]}) change in any way, based on this scenario?
            Output ONLY yes or no with no explanation, and no punctuation, all lowercase.
            """
            fallbackMessage1 = f"""
            You are evaluating a hypothetical counterfactual relationship between two events in a scenario.

            Scenario: {context}
            Variable A: {tuplePair[0]}
            Variable B: {tuplePair[1]}

            Consider a hypothetical version of the scenario in which Variable A ({tuplePair[0]}) did not occur.

            Would Variable B ({tuplePair[1]}) differ from what is described in the original scenario?

            Output ONLY yes or no with no explanation, and no punctuation, all lowercase.
            """
            fallbackMessage2 = f"""
            You are evaluating a hypothetical counterfactual relationship between two events in a scenario.

            Scenario: {context}
            Variable A: {tuplePair[1]}
            Variable B: {tuplePair[0]}

            Consider a hypothetical version of the scenario in which Variable A ({tuplePair[1]}) did not occur.

            Would Variable B ({tuplePair[0]}) differ from what is described in the original scenario?

            Output ONLY yes or no with no explanation, and no punctuation, all lowercase.
            """
        else:
            extraPairs = []
            for pair in pairs:
                for variable in pair:
                    if variable not in tuplePair and variable not in extraPairs:
                        extraPairs.append(variable)
            message1 = f"""
            You will be given a scenario and two variables from it, Variable A and Variable B. Your job is to judge whether Variable A directly causes Variable B.

            Scenario: {context}
            Variable A: {tuplePair[0]}
            Variable B: {tuplePair[1]}
            Other Variables from the scenario: {extraPairs}

            Does Variable A ({tuplePair[0]}) directly cause Variable B ({tuplePair[1]}), or is the effect of Variable A on Variable B instead transmitted through one or more of the other variables provided from the scenario?

            Output yes only if Variable A directly causes Variable B. Otherwise output no.
            Output ONLY yes or no with no explanation, and no punctuation, all lowercase.
            """
            message2 = f"""
            You will be given a scenario and two variables from it, Variable A and Variable B. Your job is to judge whether Variable A directly causes Variable B.

            Scenario: {context}
            Variable A: {tuplePair[1]}
            Variable B: {tuplePair[0]}
            Other Variables from the scenario: {extraPairs}

            Does Variable A ({tuplePair[1]}) directly cause Variable B ({tuplePair[0]}), or is the effect of Variable A on Variable B instead transmitted through one or more of the other variables provided from the scenario?

            Output yes only if Variable A directly causes Variable B. Otherwise output no.
            Output ONLY yes or no with no explanation, and no punctuation, all lowercase.
            """
            fallbackMessage1 = message1
            fallbackMessage2 = message2
        f = prompt(message1, model=model, numtoken=20)
        if f == "__SAFETY_REFUSAL__":
            print("SAFETY FALLBACK — FORWARD")
            f = prompt(fallbackMessage1, model=model, numtoken=20)
        b = prompt(message2, model=model, numtoken=20)
        if b == "__SAFETY_REFUSAL__":
            print("SAFETY FALLBACK — BACKWARD")
            b = prompt(fallbackMessage2, model=model, numtoken=20)
        if f is None or b is None:
            return "ambig"
        return f.strip() + "|" + b.strip()
    def outputDirection(self, tuplePair, unformattedOutput):
        try:
            parts = [part.strip().lower() for part in unformattedOutput.split("|")]
            if len(parts) != 2:
                return "ambig"
            if any(part not in {"yes", "no"} for part in parts):
                return "ambig"
            forward, backward = parts
        except (AttributeError, ValueError):
            return "ambig"
        if forward == "yes" and backward == "no":
            return tuplePair[0] + "->" + tuplePair[1]
        if forward == "no" and backward == "yes":
            return tuplePair[1] + "->" + tuplePair[0]
        if forward == "no" and backward == "no":
            return "noRel"
        return "ambig"
    def identifyConfounders(self, graph):
        allCauses = [item["cause"] for item in graph["edges"]]
        seen = set()
        confoundingCauses = []
        for cause in allCauses:
            if cause in seen:
                continue
            count = 0
            causeDict = {}
            effectList = []
            for edge in graph["edges"]:
                if cause == edge["cause"]:
                    count += 1
                    causeDict["cause"] = cause
                    effectList.append(edge["effect"])
                    causeDict["effects"] = effectList
            if count > 1:
                seen.add(cause)
                confoundingCauses.append(causeDict)
        return confoundingCauses
    def identifyColliders(self, graph):
        allEffects = [item["effect"] for item in graph["edges"]]
        seen = set()
        collidingCauses = []
        for effect in allEffects:
            if effect in seen:
                continue
            count = 0
            causeDict = {}
            causeList = []
            for edge in graph["edges"]:
                if effect == edge["effect"]:
                    count += 1
                    causeDict["effect"] = effect
                    causeList.append(edge["cause"])
                    causeDict["causes"] = causeList
            if count > 1:
                seen.add(effect)
                collidingCauses.append(causeDict)
        return collidingCauses
    def identifyMediators(self, graph):
        mediatorList = []
        for edge1 in graph["edges"]:
            for edge2 in graph["edges"]:
                if edge1 == edge2:
                    continue
                if edge1["effect"] == edge2["cause"]:
                    mediatorDict = {
                        "cause": edge1["cause"],
                        "mediator": edge1["effect"],
                        "effect": edge2["effect"]
                    }
                    mediatorList.append(mediatorDict)
        return mediatorList
    def checkMediation(self, model, context, start, mediator, end):
        message = f"""
            You are given a scenario and two possible representations of the causal influence from {start} to {end}.

            Scenario: {context}

            Direct:
            {start} -> {end}

            Mediated:
            {start} -> {mediator} -> {end}

            Based on the scenario, is {start}'s causal influence on {end} direct, or is it instead mediated through {mediator}?

            Output ONLY direct or mediated with no explanation, and no punctuation, all lowercase.
            """
        output = prompt(message, model, numtoken=20)
        if output is None:
            return "ambig"
        output = output.strip().lower()
        if output not in {"direct", "mediated"}:
            return "ambig"
        return output
    def refineGraph(self, graph, confounderList, colliderList, mediatorList):
        outputGraph = graph
        outputGraph["confounders"] = confounderList
        outputGraph["colliders"] = colliderList
        outputGraph["mediators"] = mediatorList
        return outputGraph
    def constructFullGraph(self, model, context, pairList, promptMediator = False, question=None, debug = False):
        outputDict = {"context": context}
        if question:
            outputDict["question"] = question
        edges = []
        for pair in pairList:
            rawOutput = self.askCausalQ(model, context, pair, pairList, promptMediator)
            direction = self.outputDirection(pair, rawOutput)
            if direction == "ambig":
                if debug:
                    print("Ambiguous relationship")
                continue
            elif direction == "noRel":
                if debug:
                    print("No relationship")
                continue
            cause, effect = direction.split("->")
            edges.append({"cause": cause, "effect": effect})
        if promptMediator:
          pass
        else:
            toCheck = []
            for firstEdge in edges:
                for secondEdge in edges:
                    if firstEdge['effect'] == secondEdge['cause']:
                        start = firstEdge['cause']
                        mediator = firstEdge['effect']
                        end = secondEdge['effect']
                        for edge in edges:
                            if edge['cause'] == start and edge['effect'] == end:
                                candidate = {
                                    'edge': edge,
                                    'mediator': mediator
                                }
                                if candidate not in toCheck:
                                    toCheck.append(candidate)
            toDelete = []
            for candidate in toCheck:
                edge = candidate['edge']
                mediator = candidate['mediator']
                result = self.checkMediation(
                    model,
                    context,
                    edge['cause'],
                    mediator,
                    edge['effect']
                )
                if result == "mediated":
                    toDelete.append(edge)
            for edge in toDelete:
                if edge in edges:
                    edges.remove(edge)
        outputDict["edges"] = edges
        confounderList, colliderList, mediatorList = (
            self.identifyConfounders(outputDict),
            self.identifyColliders(outputDict),
            self.identifyMediators(outputDict)
        )
        return self.refineGraph(
            outputDict,
            confounderList,
            colliderList,
            mediatorList
        )
    def answer(self, model, graph, choices, context, question=None, labelThirdVars=False, returnRaw=False):
        message = f"""
        You are given a context/scenario {f", Question," if question else ""} and a graph that helps represent the context.
        {"""
        The graph also marks third-variable structures:
        - Confounders: a variable that causes both the cause and effect, suggesting the direct relationship between the cause and effect isn't strong. Treat as weakening evidence for that relationship.
        - Mediators: a variable through which the causal effect travels indirectly. Treat as supporting the existence of an indirect causal relationship.
        - Colliders: a variable caused by both endpoints. Conditioning on it creates spurious correlations — treat with caution when reasoning about independence.
        """ if labelThirdVars else ""}

        Context: {context}
        {f"Question: {question}" if question else ""}
        Graph: {graph}

        Choice 1: {choices[0]}
        Choice 2: {choices[1]}
        Choice 3: {choices[2]}
        Choice 4: {choices[3]}

        From the information above, which option best answers the question?
        Output ONLY one of the following single numbers: 1, 2, 3, or 4.
        Do not output any explanation, words, punctuation, or additional text.
        """
        rawOutput = prompt(message, model=model)
        try:
            parsedOutput = int(rawOutput.strip())
            if parsedOutput not in {1, 2, 3, 4}:
                parsedOutput = -1
        except (ValueError, AttributeError):
            parsedOutput = -1
        if returnRaw:
            return parsedOutput, rawOutput
        return parsedOutput
    def runOnce(
        self,
        model,
        graph,
        context,
        choices,
        correctAnswer,
        pairList,
        question=None,
        debug=False
    ):
        selectedOption = self.answer(
            model,
            graph,
            choices,
            context,
            question=question
        )
        if debug:
            print(graph)
            print(selectedOption)
        if selectedOption == -1:
            return -1
        return int(correctAnswer == selectedOption)

In [ ]:
def unpackDatasetItem(contextItem):
    contextId = contextItem['contextId']
    dagType = contextItem['dagType']
    dag = contextItem['dag']
    passage = contextItem['passage']
    questions = contextItem['questions']
    nodes = contextItem['nodes']
    return contextId, dagType, dag, passage, questions, nodes


In [ ]:
def runBaseline(model, dataset, outputDir="/content"):
    accuracyReport = {
        'chain': 0,
        'fork': 0,
        'collider': 0,
        'diamond': 0,
        'divchain': 0,
        'convchain': 0,
        'asymmetric_divchain': 0,
        'asymmetric_convchain': 0
    }
    safeModel = model.replace("/", "__")
    outputPath = os.path.join(
        outputDir,
        f"questions_baseline_{safeModel}.json"
    )
    failedPath = os.path.join(
        outputDir,
        f"failed_baseline_{safeModel}.json"
    )
    questionResults = []
    totalQuestions = 0
    correctQuestions = 0
    invalidQuestions = 0
    apiErrors = 0
    contexts = dataset["contexts"]
    for context in contexts:
        try:
            contextId, dagType, dag, passage, questions, nodes = \
                unpackDatasetItem(context)
        except Exception as e:
            print(f"Error unpacking context: {e}")
            continue
        for qNum, (qKey, questionItem) in enumerate(
            questions.items(), 1
        ):
            questionId = makeQuestionId(contextId, qKey)
            try:
                question = questionItem["question"]
                options = list(questionItem["options"].values())
                correct = questionItem["answer"]
            except Exception as e:
                print(
                    f"Dataset error [{contextId}] Q{qNum}: {e}"
                )
                continue
            message = f"""
            You will be given a scenario, a question about the scenario, and answer options.
            Your goal is to select the most accurate answer option for the question.

            Scenario: {passage}
            Question: {question}

            OPTIONS:
            A: {options[0]}
            B: {options[1]}
            C: {options[2]}
            D: {options[3]}

            Output only a single capital letter as your answer choice representing the single option you select from above, with no explanation or redundant text.
            """
            rawOutput = None
            try:
                rawOutput = prompt(message, model)
            except Exception as e:
                print(
                    f"API ERROR | baseline | {model} | "
                    f"{questionId}: {e}"
                )
                apiErrors += 1
                record = makeQuestionRecord(
                    contextId=contextId,
                    qKey=qKey,
                    qNum=qNum,
                    dagType=dagType,
                    model=model,
                    condition="baseline",
                    correctAnswer=correct,
                    modelAnswer=None,
                    rawOutput=None,
                    status="api_error",
                    isCorrect=None,
                    error=str(e)
                )
                questionResults.append(record)
                saveQuestionResults(
                    questionResults,
                    outputPath
                )
                saveFailedQuestions(
                    questionResults,
                    failedPath
                )
                continue
            cleanedOutput = rawOutput.strip().upper()
            if (
                len(cleanedOutput) == 1
                and cleanedOutput in {"A", "B", "C", "D"}
            ):
                parsedAnswer = cleanedOutput
            else:
                parsedAnswer = None
            status, isCorrect = classifyAnswer(
                parsedAnswer,
                correct
            )
            totalQuestions += 1
            if status == "correct":
                correctQuestions += 1
                accuracyReport[dagType] += 1
            elif status == "invalid":
                invalidQuestions += 1
            record = makeQuestionRecord(
                contextId=contextId,
                qKey=qKey,
                qNum=qNum,
                dagType=dagType,
                model=model,
                condition="baseline",
                correctAnswer=correct,
                modelAnswer=parsedAnswer,
                rawOutput=rawOutput,
                status=status,
                isCorrect=isCorrect
            )
            questionResults.append(record)
            saveQuestionResults(
                questionResults,
                outputPath
            )
            if status == "invalid":
                saveFailedQuestions(
                    questionResults,
                    failedPath
                )
            if totalQuestions % 10 == 0:
                print(
                    f"Progress: {totalQuestions}/160 | "
                    f"Correct: {correctQuestions} | "
                    f"Invalid: {invalidQuestions} | "
                    f"API errors: {apiErrors}"
                )
    accuracy = (
        correctQuestions / totalQuestions
        if totalQuestions
        else 0
    )
    dagAccuracy = {
        key: value / 20
        for key, value in accuracyReport.items()
    }
    print("\n================================")
    print("BASELINE COMPLETE")
    print("================================")
    print(f"Accuracy: {correctQuestions}/{totalQuestions} = {accuracy:.2%}")
    print(f"Invalid responses: {invalidQuestions}")
    print(f"API failures needing rerun: {apiErrors}")
    print(f"Results: {outputPath}")
    print(f"Failures: {failedPath}")
    return (
        accuracy,
        dagAccuracy,
        questionResults
    )

In [ ]:
def formatGraph(nodes, edges):
    nameToLetter = {name: letter for letter, name in nodes.items()}
    lines = []
    for letter, name in nodes.items():
        lines.append(f"{letter}: {name}")
    lines.append("")
    for edge in edges:
        causeLetter = nameToLetter.get(edge['cause'], edge['cause'])
        effectLetter = nameToLetter.get(edge['effect'], edge['effect'])
        lines.append(f"{causeLetter}->{effectLetter}")
    return "\n".join(lines)

In [ ]:
def formatGraphWithThirdVars(nodes, edges):
    nameToLetter = {name: letter for letter, name in nodes.items()}
    letterEdges = []
    for edge in edges:
        cause = nameToLetter.get(edge['cause'], edge['cause'])
        effect = nameToLetter.get(edge['effect'], edge['effect'])
        letterEdges.append((cause, effect))
    outgoing = {}
    incoming = {}
    for cause, effect in letterEdges:
        if cause not in outgoing:
            outgoing[cause] = []
        outgoing[cause].append(effect)
        if effect not in incoming:
            incoming[effect] = []
        incoming[effect].append(cause)
    confounders = []
    for cause, effects in outgoing.items():
        effects = list(dict.fromkeys(effects))
        if len(effects) > 1:
            confounders.append({
                "confounder": cause,
                "effects": effects
            })
    colliders = []
    for effect, causes in incoming.items():
        causes = list(dict.fromkeys(causes))
        if len(causes) > 1:
            colliders.append({
                "collider": effect,
                "causes": causes
            })
    mediators = []
    for mediator in nodes.keys():
        if mediator in incoming and mediator in outgoing:
            for cause in incoming[mediator]:
                for effect in outgoing[mediator]:
                    mediators.append({
                        "cause": cause,
                        "mediator": mediator,
                        "effect": effect
                    })
    lines = []
    for letter, name in nodes.items():
        lines.append(f"{letter}: {name}")
    lines.append("")
    for cause, effect in letterEdges:
        lines.append(f"{cause} -> {effect}")
    lines.append("")
    lines.append("")
    lines.append("Confounders:")
    if confounders:
        for conf in confounders:
            effects = conf["effects"]
            if len(effects) == 2:
                effectsText = f"both {effects[0]} and {effects[1]}"
            else:
                effectsText = ", ".join(effects[:-1]) + f", and {effects[-1]}"
            lines.append(
                f"- {conf['confounder']} is a confounder of {effectsText}"
            )
    else:
        lines.append("- None found")
    lines.append("")
    lines.append("Mediators:")
    if mediators:
        for med in mediators:
            lines.append(
                f"- {med['mediator']} is a mediator in "
                f"{med['cause']} -> {med['mediator']} -> {med['effect']}"
            )
    else:
        lines.append("- None found")
    lines.append("")
    lines.append("Colliders:")
    if colliders:
        for coll in colliders:
            causes = coll["causes"]
            if len(causes) == 2:
                causesText = f"both {causes[0]} and {causes[1]}"
            else:
                causesText = ", ".join(causes[:-1]) + f", and {causes[-1]}"
            lines.append(
                f"- {coll['collider']} is a collider of {causesText}"
            )
    else:
        lines.append("- None found")
    lines.append("")
    thirdVarHints = """
    """
    lines.append(thirdVarHints)
    return "\n".join(lines)

In [ ]:
def runTruthyGraphs(
    model,
    dataset,
    graphs,
    thirdVarsMarked=False,
    outputDir="/content"
):
    condition = (
        "truth_labeled"
        if thirdVarsMarked
        else "truth_plain"
    )
    safeModel = model.replace("/", "__")
    outputPath = os.path.join(
        outputDir,
        f"questions_{condition}_{safeModel}.json"
    )
    failedPath = os.path.join(
        outputDir,
        f"failed_{condition}_{safeModel}.json"
    )
    accuracyReport = {
        'chain': 0,
        'fork': 0,
        'collider': 0,
        'diamond': 0,
        'divchain': 0,
        'convchain': 0,
        'asymmetric_divchain': 0,
        'asymmetric_convchain': 0
    }
    questionResults = []
    totalQuestions = 0
    correctQuestions = 0
    invalidQuestions = 0
    apiErrors = 0
    contexts = dataset["contexts"]
    for i, context in enumerate(contexts):
        try:
            contextId, dagType, dag, passage, questions, nodes = \
                unpackDatasetItem(context)
        except Exception as e:
            raise RuntimeError(
                f"Could not unpack dataset context index {i}: {e}"
            ) from e
        try:
            graph = graphs[i]["dag"]
            if thirdVarsMarked:
                graph = formatGraphWithThirdVars(
                    context["nodes"],
                    graph
                )
            else:
                graph = formatGraph(
                    context["nodes"],
                    graph
                )
        except Exception as e:
            raise RuntimeError(
                f"Truth graph formatting failed | "
                f"{contextId} | {condition}: {e}"
            ) from e
        for qNum, (qKey, questionItem) in enumerate(
            questions.items(), 1
        ):
            questionId = makeQuestionId(
                contextId,
                qKey
            )
            try:
                question = questionItem["question"]
                options = list(
                    questionItem["options"].values()
                )
                correct = questionItem["answer"]
            except Exception as e:
                raise RuntimeError(
                    f"Dataset question error | "
                    f"{contextId} | {qKey}: {e}"
                ) from e
            message = f"""
            You will be given a scenario, a graph representing the scenario, a question about the scenario, and answer options.
            The graph can be used to help understand the scenario, and can be used and is useful for answering the questions.
            {'The graph will also mark any third variable structures, more specifically confounders, colliders, and mediators' if thirdVarsMarked else ""}

            Scenario: {passage}

            Graph:
            {graph}

            Question: {question}

            OPTIONS:
            A: {options[0]}
            B: {options[1]}
            C: {options[2]}
            D: {options[3]}

            Output only a single capital letter as your answer choice representing the single option you select from above, with no explanation or redundant text.
            """
            rawOutput = None
            try:
                rawOutput = prompt(
                    message,
                    model
                )
            except Exception as e:
                apiErrors += 1
                print(
                    f"API ERROR | {condition} | "
                    f"{model} | {questionId} | {e}"
                )
                record = makeQuestionRecord(
                    contextId=contextId,
                    qKey=qKey,
                    qNum=qNum,
                    dagType=dagType,
                    model=model,
                    condition=condition,
                    correctAnswer=correct,
                    modelAnswer=None,
                    rawOutput=None,
                    status="api_error",
                    isCorrect=None,
                    error=str(e)
                )
                questionResults.append(record)
                saveQuestionResults(
                    questionResults,
                    outputPath
                )
                saveFailedQuestions(
                    questionResults,
                    failedPath
                )
                continue
            try:
                cleanedOutput = rawOutput.strip().upper()
            except:
                cleanedOutput = ""
            if (
                len(cleanedOutput) == 1
                and cleanedOutput in {"A", "B", "C", "D"}
            ):
                parsedAnswer = cleanedOutput
            else:
                parsedAnswer = None
            status, isCorrect = classifyAnswer(
                parsedAnswer,
                correct
            )
            totalQuestions += 1
            if status == "correct":
                correctQuestions += 1
                accuracyReport[dagType] += 1
            elif status == "invalid":
                invalidQuestions += 1
                print(
                    f"INVALID | {condition} | "
                    f"{model} | {questionId} | "
                    f"raw={repr(rawOutput)}"
                )
            record = makeQuestionRecord(
                contextId=contextId,
                qKey=qKey,
                qNum=qNum,
                dagType=dagType,
                model=model,
                condition=condition,
                correctAnswer=correct,
                modelAnswer=parsedAnswer,
                rawOutput=rawOutput,
                status=status,
                isCorrect=isCorrect,
                error=None
            )
            questionResults.append(record)
            saveQuestionResults(
                questionResults,
                outputPath
            )
            if status == "invalid":
                saveFailedQuestions(
                    questionResults,
                    failedPath
                )
            if totalQuestions % 10 == 0:
                print(
                    f"{condition} | "
                    f"{totalQuestions}/160 completed | "
                    f"Correct: {correctQuestions} | "
                    f"Invalid: {invalidQuestions} | "
                    f"API errors: {apiErrors}"
                )
    totalAccuracy = (
        correctQuestions / totalQuestions
        if totalQuestions
        else 0
    )
    dagAccuracy = {
        key: value / 20
        for key, value
        in accuracyReport.items()
    }
    saveFailedQuestions(
        questionResults,
        failedPath
    )
    print("\n======================================")
    print(condition.upper())
    print("======================================")
    print(
        f"Accuracy: "
        f"{correctQuestions}/{totalQuestions} "
        f"= {totalAccuracy:.2%}"
    )
    print(
        f"Invalid formatting responses: "
        f"{invalidQuestions}"
    )
    print(
        f"API failures requiring rerun: "
        f"{apiErrors}"
    )
    print(
        f"Total records saved: "
        f"{len(questionResults)}"
    )
    print(
        f"Results file: {outputPath}"
    )
    print(
        f"Failures file: {failedPath}"
    )
    return (
        totalAccuracy,
        dagAccuracy,
        questionResults
    )

In [ ]:
def runFullGraphs(
    model,
    dataset,
    promptMediator=False,
    checkpointDir="/content"
):
    accuracyTemplate = {
        'chain': 0,
        'fork': 0,
        'collider': 0,
        'diamond': 0,
        'divchain': 0,
        'convchain': 0,
        'asymmetric_divchain': 0,
        'asymmetric_convchain': 0
    }
    contexts = dataset["contexts"]
    mode = "mediator" if promptMediator else "normal"
    safeModel = model.replace("/", "__")
    datasetFingerprint = hashlib.sha256(
        json.dumps(dataset, sort_keys=True).encode()
    ).hexdigest()[:12]
    checkpointPath = os.path.join(
        checkpointDir,
        f"fullgraph_{mode}_{safeModel}_{datasetFingerprint}.json"
    )
    failedPath = os.path.join(
        checkpointDir,
        f"failed_fullgraph_{mode}_{safeModel}_{datasetFingerprint}.json"
    )
    def saveCheckpoint(state):
        tempPath = checkpointPath + ".tmp"
        with open(tempPath, "w") as f:
            json.dump(state, f, indent=2)
        os.replace(tempPath, checkpointPath)
    if os.path.exists(checkpointPath):
        with open(checkpointPath, "r") as f:
            state = json.load(f)
        if (
            state["model"] != model
            or state["promptMediator"] != promptMediator
            or state["datasetFingerprint"] != datasetFingerprint
        ):
            raise RuntimeError(
                "Checkpoint does not match this experiment."
            )
        state.setdefault("questionResults", [])
        state.setdefault("invalidResponses", [])
        state.setdefault("apiErrors", [])
        print("\nRESUMING CHECKPOINT")
        print("Model:", model)
        print("Mode:", mode)
        print("Next context:", state["contextIndex"] + 1)
        print("Next question:", state["questionIndex"] + 1)
        print("Stage:", state["stage"])
        if state.get("complete", False):
            print("\nCheckpoint already complete.")
            return (
                state["totalAccuracyPlain"],
                state["dagAccuracyPlain"],
                state["totalAccuracyLabeled"],
                state["dagAccuracyLabeled"],
                state["generatedGraphs"],
                state["questionResults"]
            )
    else:
        state = {
            "model": model,
            "promptMediator": promptMediator,
            "datasetFingerprint": datasetFingerprint,
            "contextIndex": 0,
            "questionIndex": 0,
            "stage": "plain",
            "currentContextId": None,
            "currentGraph": None,
            "accuracyReportPlain":
                accuracyTemplate.copy(),
            "accuracyReportLabeled":
                accuracyTemplate.copy(),
            "totalQuestionsPlain": 0,
            "correctQuestionsPlain": 0,
            "totalAttemptedPlain": 0,
            "totalQuestionsLabeled": 0,
            "correctQuestionsLabeled": 0,
            "totalAttemptedLabeled": 0,
            "generatedGraphs": [],
            "questionResults": [],
            "invalidResponses": [],
            "apiErrors": [],
            "complete": False
        }
        saveCheckpoint(state)
    graphBuilder = fullGraph()
    answerMap = {
        'A': 1,
        'B': 2,
        'C': 3,
        'D': 4
    }
    for contextIndex in range(
        state["contextIndex"],
        len(contexts)
    ):
        context = contexts[contextIndex]
        contextId, dagType, dag, passage, questions, nodes = \
            unpackDatasetItem(context)
        if state["currentGraph"] is None:
            pairList = list(
                itertools.combinations(
                    nodes.values(),
                    2
                )
            )
            try:
                graph = graphBuilder.constructFullGraph(
                    model,
                    passage,
                    pairList,
                    promptMediator=promptMediator
                )
            except Exception as e:
                print(
                    f"GRAPH API ERROR | "
                    f"{mode} | {model} | "
                    f"{contextId}: {e}"
                )
                graphFailure = {
                    "context_id": contextId,
                    "model": model,
                    "graph_method": f"full_{mode}",
                    "status": "api_error",
                    "error": str(e)
                }
                state["apiErrors"].append(
                    graphFailure
                )
                saveCheckpoint(state)
                state["contextIndex"] = (
                    contextIndex + 1
                )
                state["questionIndex"] = 0
                state["stage"] = "plain"
                state["currentContextId"] = None
                state["currentGraph"] = None
                saveCheckpoint(state)
                continue
            state["currentContextId"] = contextId
            state["currentGraph"] = graph
            state["questionIndex"] = 0
            state["stage"] = "plain"
            state["generatedGraphs"].append({
                "contextId": contextId,
                "graph": graph["edges"]
            })
            saveCheckpoint(state)
        else:
            if (
                state["currentContextId"]
                != contextId
            ):
                raise RuntimeError(
                    f"Checkpoint context mismatch: "
                    f"{state['currentContextId']} "
                    f"vs {contextId}"
                )
            graph = state["currentGraph"]
        plainGraph = {
            "context": graph["context"],
            "edges": graph["edges"]
        }
        questionItems = list(
            questions.items()
        )
        for questionIndex in range(
            state["questionIndex"],
            len(questionItems)
        ):
            qKey, questionItem = \
                questionItems[questionIndex]
            qNum = questionIndex + 1
            questionId = makeQuestionId(
                contextId,
                qKey
            )
            question = questionItem["question"]
            options = list(
                questionItem["options"].values()
            )
            correct = answerMap[
                questionItem["answer"]
            ]
            if state["stage"] == "plain":
                condition = (
                    "full_nocleanup_plain"
                    if promptMediator
                    else "full_plain"
                )
                rawOutputPlain = None
                try:
                    (
                        selectedOptionPlain,
                        rawOutputPlain
                    ) = graphBuilder.answer(
                        model,
                        plainGraph,
                        options,
                        passage,
                        question=question,
                        labelThirdVars=False,
                        returnRaw=True
                    )
                except Exception as e:
                    print(
                        f"API ERROR | "
                        f"{condition} | "
                        f"{model} | "
                        f"{questionId} | {e}"
                    )
                    record = makeQuestionRecord(
                        contextId=contextId,
                        qKey=qKey,
                        qNum=qNum,
                        dagType=dagType,
                        model=model,
                        condition=condition,
                        correctAnswer=correct,
                        modelAnswer=None,
                        rawOutput=None,
                        status="api_error",
                        isCorrect=None,
                        error=str(e)
                    )
                    state[
                        "questionResults"
                    ].append(record)
                    state[
                        "apiErrors"
                    ].append(record)
                    state["stage"] = "labeled"
                    saveCheckpoint(state)
                else:
                    state[
                        "totalAttemptedPlain"
                    ] += 1
                    if selectedOptionPlain == -1:
                        status = "invalid"
                        isCorrect = 0
                        parsedAnswer = None
                        state[
                            "totalQuestionsPlain"
                        ] += 1
                        state[
                            "invalidResponses"
                        ].append({
                            "contextId": contextId,
                            "qKey": qKey,
                            "qNum": qNum,
                            "condition": condition,
                            "raw_output":
                                rawOutputPlain
                        })
                    else:
                        parsedAnswer = (
                            selectedOptionPlain
                        )
                        state[
                            "totalQuestionsPlain"
                        ] += 1
                        if (
                            selectedOptionPlain
                            == correct
                        ):
                            status = "correct"
                            isCorrect = 1
                            state[
                                "correctQuestionsPlain"
                            ] += 1
                            state[
                                "accuracyReportPlain"
                            ][dagType] += 1
                        else:
                            status = "wrong"
                            isCorrect = 0
                    record = makeQuestionRecord(
                        contextId=contextId,
                        qKey=qKey,
                        qNum=qNum,
                        dagType=dagType,
                        model=model,
                        condition=condition,
                        correctAnswer=correct,
                        modelAnswer=parsedAnswer,
                        rawOutput=rawOutputPlain,
                        status=status,
                        isCorrect=isCorrect,
                        error=None
                    )
                    state[
                        "questionResults"
                    ].append(record)
                    state["stage"] = "labeled"
                    saveCheckpoint(state)
            if state["stage"] == "labeled":
                condition = (
                    "full_nocleanup_labeled"
                    if promptMediator
                    else "full_labeled"
                )
                rawOutputLabeled = None
                try:
                    (
                        selectedOptionLabeled,
                        rawOutputLabeled
                    ) = graphBuilder.answer(
                        model,
                        graph,
                        options,
                        passage,
                        question=question,
                        labelThirdVars=True,
                        returnRaw=True
                    )
                except Exception as e:
                    print(
                        f"API ERROR | "
                        f"{condition} | "
                        f"{model} | "
                        f"{questionId} | {e}"
                    )
                    record = makeQuestionRecord(
                        contextId=contextId,
                        qKey=qKey,
                        qNum=qNum,
                        dagType=dagType,
                        model=model,
                        condition=condition,
                        correctAnswer=correct,
                        modelAnswer=None,
                        rawOutput=None,
                        status="api_error",
                        isCorrect=None,
                        error=str(e)
                    )
                    state[
                        "questionResults"
                    ].append(record)
                    state[
                        "apiErrors"
                    ].append(record)
                else:
                    state[
                        "totalAttemptedLabeled"
                    ] += 1
                    if selectedOptionLabeled == -1:
                        status = "invalid"
                        isCorrect = 0
                        parsedAnswer = None
                        state[
                            "totalQuestionsLabeled"
                        ] += 1
                        state[
                            "invalidResponses"
                        ].append({
                            "contextId": contextId,
                            "qKey": qKey,
                            "qNum": qNum,
                            "condition": condition,
                            "raw_output":
                                rawOutputLabeled
                        })
                    else:
                        parsedAnswer = (
                            selectedOptionLabeled
                        )
                        state[
                            "totalQuestionsLabeled"
                        ] += 1
                        if (
                            selectedOptionLabeled
                            == correct
                        ):
                            status = "correct"
                            isCorrect = 1
                            state[
                                "correctQuestionsLabeled"
                            ] += 1
                            state[
                                "accuracyReportLabeled"
                            ][dagType] += 1
                        else:
                            status = "wrong"
                            isCorrect = 0
                    record = makeQuestionRecord(
                        contextId=contextId,
                        qKey=qKey,
                        qNum=qNum,
                        dagType=dagType,
                        model=model,
                        condition=condition,
                        correctAnswer=correct,
                        modelAnswer=parsedAnswer,
                        rawOutput=rawOutputLabeled,
                        status=status,
                        isCorrect=isCorrect,
                        error=None
                    )
                    state[
                        "questionResults"
                    ].append(record)
                state["questionIndex"] = (
                    questionIndex + 1
                )
                state["stage"] = "plain"
                saveCheckpoint(state)
        state["contextIndex"] = (
            contextIndex + 1
        )
        state["questionIndex"] = 0
        state["stage"] = "plain"
        state["currentContextId"] = None
        state["currentGraph"] = None
        saveCheckpoint(state)
    totalAccuracyPlain = (
        state["correctQuestionsPlain"]
        / state["totalQuestionsPlain"]
        if state["totalQuestionsPlain"]
        else 0
    )
    totalAccuracyLabeled = (
        state["correctQuestionsLabeled"]
        / state["totalQuestionsLabeled"]
        if state["totalQuestionsLabeled"]
        else 0
    )
    dagAccuracyPlain = {
        key: value / 20
        for key, value
        in state[
            "accuracyReportPlain"
        ].items()
    }
    dagAccuracyLabeled = {
        key: value / 20
        for key, value
        in state[
            "accuracyReportLabeled"
        ].items()
    }
    state[
        "totalAccuracyPlain"
    ] = totalAccuracyPlain
    state[
        "totalAccuracyLabeled"
    ] = totalAccuracyLabeled
    state[
        "dagAccuracyPlain"
    ] = dagAccuracyPlain
    state[
        "dagAccuracyLabeled"
    ] = dagAccuracyLabeled
    state["complete"] = True
    saveCheckpoint(state)
    failedRecords = [
        x for x
        in state["questionResults"]
        if x["status"]
        in {"api_error", "invalid"}
    ]
    tempPath = failedPath + ".tmp"
    with open(tempPath, "w") as f:
        json.dump(
            failedRecords,
            f,
            indent=2
        )
    os.replace(
        tempPath,
        failedPath
    )
    print("\n======================================")
    print(f"FULLGRAPH {mode.upper()} COMPLETE")
    print("======================================")
    print(
        f"Plain: "
        f"{state['correctQuestionsPlain']}/"
        f"{state['totalQuestionsPlain']} "
        f"= {totalAccuracyPlain:.2%}"
    )
    print(
        f"Labeled: "
        f"{state['correctQuestionsLabeled']}/"
        f"{state['totalQuestionsLabeled']} "
        f"= {totalAccuracyLabeled:.2%}"
    )
    apiQuestionErrors = [
        x for x
        in state["questionResults"]
        if x["status"] == "api_error"
    ]
    print(
        "Invalid responses:",
        len(state["invalidResponses"])
    )
    print(
        "Question API failures requiring rerun:",
        len(apiQuestionErrors)
    )
    print(
        "Question records:",
        len(state["questionResults"])
    )
    print(
        "Generated graphs:",
        len(state["generatedGraphs"])
    )
    return (
        totalAccuracyPlain,
        dagAccuracyPlain,
        totalAccuracyLabeled,
        dagAccuracyLabeled,
        state["generatedGraphs"],
        state["questionResults"]
    )

In [ ]:
def constructQuickGraph(model, context, nodes, labelThirdVars=False):
    labels = ["a", "b", "c", "d"]
    trueLabels = ["A", "B", "C", "D"]
    labelMap = dict(zip(labels, trueLabels))
    events = ", ".join(
        f"{label} = {nodes[labelMap[label]]}"
        for label in labels
    )
    message = f"""
      You will be given a scenario and 4 events from that scenario.

      Scenario: {context}
      Events: {events}

      Create the causal graph that is best supported by the scenario.

      Include only DIRECT causal relationships between the listed events.
      Do NOT include indirect relationships that occur through another listed event.

      Represent each edge exactly as:
      cause:effect

      Separate multiple edges with commas.

      Example output:
      a:b, b:c, c:d

      Output ONLY the graph in this format, with no explanation or extra text.
      """
    output = prompt(message, model=model, numtoken=100)
    if output is None:
        return {
            "context": context,
            "edges": []
        }
    pairs = re.findall(
        r'([a-dA-D])\s*:\s*([a-dA-D])',
        output
    )
    edges = []
    seen = set()
    for cause, effect in pairs:
        cause = cause.lower()
        effect = effect.lower()
        if cause not in labelMap or effect not in labelMap:
            continue
        trueCause = labelMap[cause]
        trueEffect = labelMap[effect]
        edgeKey = (trueCause, trueEffect)
        if edgeKey in seen:
            continue
        seen.add(edgeKey)
        edges.append({
            "cause": nodes[trueCause],
            "effect": nodes[trueEffect]
        })
    graph = {
        "context": context,
        "edges": edges
    }
    if labelThirdVars:
        graphBuilder = fullGraph()
        confounderList = graphBuilder.identifyConfounders(graph)
        colliderList = graphBuilder.identifyColliders(graph)
        mediatorList = graphBuilder.identifyMediators(graph)
        graph = graphBuilder.refineGraph(
            graph,
            confounderList,
            colliderList,
            mediatorList
        )
    return graph

In [ ]:
def runQuickGraphs(model, dataset, outputDir="/content"):
    accuracyReportPlain = {
        'chain': 0,
        'fork': 0,
        'collider': 0,
        'diamond': 0,
        'divchain': 0,
        'convchain': 0,
        'asymmetric_divchain': 0,
        'asymmetric_convchain': 0
    }
    accuracyReportLabeled = {
        'chain': 0,
        'fork': 0,
        'collider': 0,
        'diamond': 0,
        'divchain': 0,
        'convchain': 0,
        'asymmetric_divchain': 0,
        'asymmetric_convchain': 0
    }
    safeModel = model.replace("/", "__")
    outputPath = os.path.join(
        outputDir,
        f"questions_quick_{safeModel}.json"
    )
    failedPath = os.path.join(
        outputDir,
        f"failed_quick_{safeModel}.json"
    )
    graphPath = os.path.join(
        outputDir,
        f"graphs_quick_{safeModel}.json"
    )
    questionResults = []
    generatedGraphs = []
    graphErrors = []
    totalQuestionsPlain = 0
    correctQuestionsPlain = 0
    totalQuestionsLabeled = 0
    correctQuestionsLabeled = 0
    invalidPlain = 0
    invalidLabeled = 0
    apiErrors = 0
    graphBuilder = fullGraph()
    answerMap = {
        'A': 1,
        'B': 2,
        'C': 3,
        'D': 4
    }
    contexts = dataset["contexts"]
    def saveAll():
        saveQuestionResults(
            questionResults,
            outputPath
        )
        saveFailedQuestions(
            questionResults,
            failedPath
        )
        tempPath = graphPath + ".tmp"
        with open(tempPath, "w") as f:
            json.dump(
                {
                    "model": model,
                    "generatedGraphs": generatedGraphs,
                    "graphErrors": graphErrors
                },
                f,
                indent=2
            )
        os.replace(
            tempPath,
            graphPath
        )
    for context in contexts:
        try:
            (
                contextId,
                dagType,
                dag,
                passage,
                questions,
                nodes
            ) = unpackDatasetItem(context)
        except Exception as e:
            print(
                f"DATASET CONTEXT ERROR: {e}"
            )
            continue
        try:
            graph = constructQuickGraph(
                model,
                passage,
                nodes
            )
        except Exception as e:
            print(
                f"GRAPH API ERROR | quick | "
                f"{model} | {contextId} | {e}"
            )
            graphErrors.append({
                "context_id": contextId,
                "model": model,
                "graph_method": "quick",
                "status": "api_error",
                "error": str(e)
            })
            for qNum, (qKey, questionItem) in enumerate(
                questions.items(), 1
            ):
                correct = answerMap[
                    questionItem["answer"]
                ]
                for condition in [
                    "quick_plain",
                    "quick_labeled"
                ]:
                    record = makeQuestionRecord(
                        contextId=contextId,
                        qKey=qKey,
                        qNum=qNum,
                        dagType=dagType,
                        model=model,
                        condition=condition,
                        correctAnswer=correct,
                        modelAnswer=None,
                        rawOutput=None,
                        status="api_error",
                        isCorrect=None,
                        error=(
                            "graph_construction_failed: "
                            + str(e)
                        )
                    )
                    questionResults.append(
                        record
                    )
                    apiErrors += 1
            saveAll()
            continue
        generatedGraphs.append({
            "contextId": contextId,
            "graph": graph["edges"]
        })
        saveAll()
        plainGraph = {
            "context": graph["context"],
            "edges": graph["edges"]
        }
        labeledGraph = {
            "context": graph["context"],
            "edges": graph["edges"]
        }
        confounderList = \
            graphBuilder.identifyConfounders(
                labeledGraph
            )
        colliderList = \
            graphBuilder.identifyColliders(
                labeledGraph
            )
        mediatorList = \
            graphBuilder.identifyMediators(
                labeledGraph
            )
        labeledGraph = graphBuilder.refineGraph(
            labeledGraph,
            confounderList,
            colliderList,
            mediatorList
        )
        for qNum, (qKey, questionItem) in enumerate(
            questions.items(), 1
        ):
            questionId = makeQuestionId(
                contextId,
                qKey
            )
            try:
                question = questionItem[
                    "question"
                ]
                options = list(
                    questionItem[
                        "options"
                    ].values()
                )
                correct = answerMap[
                    questionItem["answer"]
                ]
            except Exception as e:
                raise RuntimeError(
                    f"Dataset question error | "
                    f"{contextId} | {qKey}: {e}"
                ) from e
            rawOutputPlain = None
            try:
                (
                    selectedOptionPlain,
                    rawOutputPlain
                ) = graphBuilder.answer(
                    model,
                    plainGraph,
                    options,
                    passage,
                    question=question,
                    labelThirdVars=False,
                    returnRaw=True
                )
            except Exception as e:
                print(
                    f"API ERROR | quick_plain | "
                    f"{model} | {questionId} | {e}"
                )
                apiErrors += 1
                record = makeQuestionRecord(
                    contextId=contextId,
                    qKey=qKey,
                    qNum=qNum,
                    dagType=dagType,
                    model=model,
                    condition="quick_plain",
                    correctAnswer=correct,
                    modelAnswer=None,
                    rawOutput=None,
                    status="api_error",
                    isCorrect=None,
                    error=str(e)
                )
                questionResults.append(
                    record
                )
            else:
                totalQuestionsPlain += 1
                if selectedOptionPlain == -1:
                    parsedAnswerPlain = None
                    statusPlain = "invalid"
                    correctPlain = 0
                    invalidPlain += 1
                    print(
                        f"INVALID | quick_plain | "
                        f"{model} | {questionId} | "
                        f"raw={repr(rawOutputPlain)}"
                    )
                elif selectedOptionPlain == correct:
                    parsedAnswerPlain = \
                        selectedOptionPlain
                    statusPlain = "correct"
                    correctPlain = 1
                    correctQuestionsPlain += 1
                    accuracyReportPlain[
                        dagType
                    ] += 1
                else:
                    parsedAnswerPlain = \
                        selectedOptionPlain
                    statusPlain = "wrong"
                    correctPlain = 0
                record = makeQuestionRecord(
                    contextId=contextId,
                    qKey=qKey,
                    qNum=qNum,
                    dagType=dagType,
                    model=model,
                    condition="quick_plain",
                    correctAnswer=correct,
                    modelAnswer=parsedAnswerPlain,
                    rawOutput=rawOutputPlain,
                    status=statusPlain,
                    isCorrect=correctPlain,
                    error=None
                )
                questionResults.append(
                    record
                )
            saveAll()
            rawOutputLabeled = None
            try:
                (
                    selectedOptionLabeled,
                    rawOutputLabeled
                ) = graphBuilder.answer(
                    model,
                    labeledGraph,
                    options,
                    passage,
                    question=question,
                    labelThirdVars=True,
                    returnRaw=True
                )
            except Exception as e:
                print(
                    f"API ERROR | quick_labeled | "
                    f"{model} | {questionId} | {e}"
                )
                apiErrors += 1
                record = makeQuestionRecord(
                    contextId=contextId,
                    qKey=qKey,
                    qNum=qNum,
                    dagType=dagType,
                    model=model,
                    condition="quick_labeled",
                    correctAnswer=correct,
                    modelAnswer=None,
                    rawOutput=None,
                    status="api_error",
                    isCorrect=None,
                    error=str(e)
                )
                questionResults.append(
                    record
                )
            else:
                totalQuestionsLabeled += 1
                if selectedOptionLabeled == -1:
                    parsedAnswerLabeled = None
                    statusLabeled = "invalid"
                    correctLabeled = 0
                    invalidLabeled += 1
                    print(
                        f"INVALID | quick_labeled | "
                        f"{model} | {questionId} | "
                        f"raw={repr(rawOutputLabeled)}"
                    )
                elif selectedOptionLabeled == correct:
                    parsedAnswerLabeled = \
                        selectedOptionLabeled
                    statusLabeled = "correct"
                    correctLabeled = 1
                    correctQuestionsLabeled += 1
                    accuracyReportLabeled[
                        dagType
                    ] += 1
                else:
                    parsedAnswerLabeled = \
                        selectedOptionLabeled
                    statusLabeled = "wrong"
                    correctLabeled = 0
                record = makeQuestionRecord(
                    contextId=contextId,
                    qKey=qKey,
                    qNum=qNum,
                    dagType=dagType,
                    model=model,
                    condition="quick_labeled",
                    correctAnswer=correct,
                    modelAnswer=parsedAnswerLabeled,
                    rawOutput=rawOutputLabeled,
                    status=statusLabeled,
                    isCorrect=correctLabeled,
                    error=None
                )
                questionResults.append(
                    record
                )
            saveAll()
        print(
            f"{contextId} complete | "
            f"Graphs: {len(generatedGraphs)}/32 | "
            f"Question records: "
            f"{len(questionResults)}/320"
        )
    totalAccuracyPlain = (
        correctQuestionsPlain /
        totalQuestionsPlain
        if totalQuestionsPlain
        else 0
    )
    totalAccuracyLabeled = (
        correctQuestionsLabeled /
        totalQuestionsLabeled
        if totalQuestionsLabeled
        else 0
    )
    dagAccuracyPlain = {
        key: value / 20
        for key, value
        in accuracyReportPlain.items()
    }
    dagAccuracyLabeled = {
        key: value / 20
        for key, value
        in accuracyReportLabeled.items()
    }
    saveAll()
    needsRerun = [
        x for x
        in questionResults
        if x["correct"] is None
    ]
    invalidResponses = [
        x for x
        in questionResults
        if x["status"] == "invalid"
    ]
    print("\n======================================")
    print("QUICKGRAPH COMPLETE")
    print("======================================")
    print(
        f"Plain: "
        f"{correctQuestionsPlain}/"
        f"{totalQuestionsPlain} "
        f"= {totalAccuracyPlain:.2%}"
    )
    print(
        f"Labeled: "
        f"{correctQuestionsLabeled}/"
        f"{totalQuestionsLabeled} "
        f"= {totalAccuracyLabeled:.2%}"
    )
    print(
        f"Generated graphs: "
        f"{len(generatedGraphs)}/32"
    )
    print(
        f"Question records: "
        f"{len(questionResults)}/320"
    )
    print(
        f"API/graph failures needing rerun: "
        f"{len(needsRerun)}"
    )
    print(
        f"Formatting failures: "
        f"{len(invalidResponses)}"
    )
    return (
        totalAccuracyPlain,
        dagAccuracyPlain,
        totalAccuracyLabeled,
        dagAccuracyLabeled,
        generatedGraphs,
        questionResults
    )

In [ ]:
def evaluateGraphReconstruction(generatedGraphs, trueDAGs):
    trueGraphMap = {
        item['contextId']: item['dag']
        for item in trueDAGs
    }
    exactMatches = 0
    totalGraphs = 0
    totalTP = 0
    totalFP = 0
    totalFN = 0
    totalEditDistance = 0
    for generated in generatedGraphs:
        contextId = generated['contextId']
        if contextId not in trueGraphMap:
            continue
        predictedEdges = {
            (edge['cause'], edge['effect'])
            for edge in generated['graph']
        }
        trueEdges = {
            (edge['cause'], edge['effect'])
            for edge in trueGraphMap[contextId]
        }
        totalGraphs += 1
        if predictedEdges == trueEdges:
            exactMatches += 1
        tp = len(predictedEdges & trueEdges)
        fp = len(predictedEdges - trueEdges)
        fn = len(trueEdges - predictedEdges)
        totalTP += tp
        totalFP += fp
        totalFN += fn
        totalEditDistance += fp + fn
    exactAccuracy = exactMatches / totalGraphs if totalGraphs else 0
    precision = (
        totalTP / (totalTP + totalFP)
        if totalTP + totalFP else 0
    )
    recall = (
        totalTP / (totalTP + totalFN)
        if totalTP + totalFN else 0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )
    meanEditDistance = (
        totalEditDistance / totalGraphs
        if totalGraphs else 0
    )
    if totalGraphs:
        z = 1.96
        p = exactAccuracy
        n = totalGraphs
        denominator = 1 + (z ** 2 / n)
        center = (
            p + (z ** 2 / (2 * n))
        ) / denominator
        margin = (
            z * (
                (
                    p * (1 - p) / n
                    + z ** 2 / (4 * n ** 2)
                ) ** 0.5
            )
        ) / denominator
        wilsonLower = center - margin
        wilsonUpper = center + margin
    else:
        wilsonLower = 0
        wilsonUpper = 0
    return {
        'exactMatches': exactMatches,
        'totalGraphs': totalGraphs,
        'exactAccuracy': exactAccuracy,
        'wilsonLower': wilsonLower,
        'wilsonUpper': wilsonUpper,
        'truePositives': totalTP,
        'falsePositives': totalFP,
        'falseNegatives': totalFN,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'meanEditDistance': meanEditDistance
    }

In [ ]:
testEdges = [
      {
        "cause": "Euclid surveying the distant universe",
        "effect": "31 quasars being discovered"
      },
      {
        "cause": "31 quasars being discovered",
        "effect": "2 ancient quasars being identified using redlight technology"
      },
      {
        "cause": "2 ancient quasars being identified using redlight technology",
        "effect": "Understanding of early black hole growth advancing"
      }
    ]
testNodes = {
        "A": "Euclid surveying the distant universe",
        "B": "31 quasars being discovered",
        "C": "2 ancient quasars being identified using redlight technology",
        "D": "Understanding of early black hole growth advancing"
      }

In [ ]:
def runFinalEval(
    model,
    dataset,
    trueDAGs,
    runBaselineCondition=False,
    runTruthCondition=False,
    runFullNormalCondition=False,
    runFullMediatorCondition=False,
    runQuickCondition=False
):
    results = {}
    if runBaselineCondition:
        print("\n==============================")
        print("BASELINE")
        print("==============================")
        try:
            totalAccuracy, dagAccuracy, questionResults = runBaseline(
                model,
                dataset
            )
            results["baseline"] = {
                "totalAccuracy": totalAccuracy,
                "dagAccuracy": dagAccuracy,
                "questionResults": questionResults
            }
            print("Baseline Accuracy:", totalAccuracy)
            print("Baseline DAG Accuracy:", dagAccuracy)
        except Exception as e:
            results["baseline"] = {
                "error": str(e)
            }
            print(f"ERROR | baseline | {model} | {e}")
    if runTruthCondition:
        print("\n==============================")
        print("TRUTH GRAPH - PLAIN")
        print("==============================")
        try:
            plainAccuracy, plainDagAccuracy, plainQuestionResults = runTruthyGraphs(
                model,
                dataset,
                trueDAGs,
                thirdVarsMarked=False
            )
            results["truth_plain"] = {
                "totalAccuracy": plainAccuracy,
                "dagAccuracy": plainDagAccuracy,
                "plainQuestionResults": plainQuestionResults
            }
            print("Truth Plain Accuracy:", plainAccuracy)
            print("Truth Plain DAG Accuracy:", plainDagAccuracy)
        except Exception as e:
            results["truth_plain"] = {
                "error": str(e)
            }
            print(f"ERROR | truth_plain | {model} | {e}")
        print("\n==============================")
        print("TRUTH GRAPH - LABELED")
        print("==============================")
        try:
            labeledAccuracy, labeledDagAccuracy, labeledQuestionResults = runTruthyGraphs(
                model,
                dataset,
                trueDAGs,
                thirdVarsMarked=True
            )
            results["truth_labeled"] = {
                "totalAccuracy": labeledAccuracy,
                "dagAccuracy": labeledDagAccuracy,
                "labeledQuestionResults": labeledQuestionResults
            }
            print("Truth Labeled Accuracy:", labeledAccuracy)
            print("Truth Labeled DAG Accuracy:", labeledDagAccuracy)
        except Exception as e:
            results["truth_labeled"] = {
                "error": str(e)
            }
            print(f"ERROR | truth_labeled | {model} | {e}")
    if runFullNormalCondition:
        print("\n==============================")
        print("FULLGRAPH - NORMAL")
        print("==============================")
        try:
            (
                plainAccuracy,
                plainDagAccuracy,
                labeledAccuracy,
                labeledDagAccuracy,
                generatedGraphs,
                questionResults
            ) = runFullGraphs(
                model,
                dataset,
                promptMediator=False
            )
            graphAccuracy = evaluateGraphReconstruction(
                generatedGraphs,
                trueDAGs
            )
            results["full_normal"] = {
                "plainAccuracy": plainAccuracy,
                "plainDagAccuracy": plainDagAccuracy,
                "labeledAccuracy": labeledAccuracy,
                "labeledDagAccuracy": labeledDagAccuracy,
                "graphAccuracy": graphAccuracy,
                "generatedGraphs": generatedGraphs,
                "questionResults": questionResults
            }
            print("Plain Accuracy:", plainAccuracy)
            print("Labeled Accuracy:", labeledAccuracy)
            print("Plain DAG Accuracy:", plainDagAccuracy)
            print("Labeled DAG Accuracy:", labeledDagAccuracy)
            print("Graph Reconstruction:", graphAccuracy)
        except Exception as e:
            results["full_normal"] = {
                "error": str(e)
            }
            print(f"ERROR | full_normal | {model} | {e}")
    if runFullMediatorCondition:
        print("\n==============================")
        print("FULLGRAPH - PROMPT MEDIATOR")
        print("==============================")
        try:
            (
                plainAccuracy,
                plainDagAccuracy,
                labeledAccuracy,
                labeledDagAccuracy,
                generatedGraphs
            ) = runFullGraphs(
                model,
                dataset,
                promptMediator=True
            )
            graphAccuracy = evaluateGraphReconstruction(
                generatedGraphs,
                trueDAGs
            )
            results["full_mediator"] = {
                "plainAccuracy": plainAccuracy,
                "plainDagAccuracy": plainDagAccuracy,
                "labeledAccuracy": labeledAccuracy,
                "labeledDagAccuracy": labeledDagAccuracy,
                "graphAccuracy": graphAccuracy,
                "generatedGraphs": generatedGraphs
            }
            print("Plain Accuracy:", plainAccuracy)
            print("Labeled Accuracy:", labeledAccuracy)
            print("Plain DAG Accuracy:", plainDagAccuracy)
            print("Labeled DAG Accuracy:", labeledDagAccuracy)
            print("Graph Reconstruction:", graphAccuracy)
        except Exception as e:
            results["full_mediator"] = {
                "error": str(e)
            }
            print(f"ERROR | full_mediator | {model} | {e}")
    if runQuickCondition:
        print("\n==============================")
        print("QUICKGRAPH")
        print("==============================")
        try:
            (
                plainAccuracy,
                plainDagAccuracy,
                labeledAccuracy,
                labeledDagAccuracy,
                generatedGraphs,
                questionResults
            ) = runQuickGraphs(
                model,
                dataset
            )
            graphAccuracy = evaluateGraphReconstruction(
                generatedGraphs,
                trueDAGs
            )
            results["quick"] = {
                "plainAccuracy": plainAccuracy,
                "plainDagAccuracy": plainDagAccuracy,
                "labeledAccuracy": labeledAccuracy,
                "labeledDagAccuracy": labeledDagAccuracy,
                "graphAccuracy": graphAccuracy,
                "generatedGraphs": generatedGraphs,
                "questionResults": questionResults
            }
            print("Plain Accuracy:", plainAccuracy)
            print("Labeled Accuracy:", labeledAccuracy)
            print("Plain DAG Accuracy:", plainDagAccuracy)
            print("Labeled DAG Accuracy:", labeledDagAccuracy)
            print("Graph Reconstruction:", graphAccuracy)
        except Exception as e:
            results["quick"] = {
                "error": str(e)
            }
            print(f"ERROR | quick | {model} | {e}")
    return results

In [ ]:
import os
import glob
import shutil
folder = "/content/study2_results"
os.makedirs(folder, exist_ok=True)
patterns = [
    "/content/questions_*.json",
    "/content/failed_*.json",
    "/content/fullgraph_*.json",
    "/content/graphs_quick_*.json"
]
for pattern in patterns:
    for file in glob.glob(pattern):
        shutil.move(file, folder)
print("Moved files to:", folder)
for file in os.listdir(folder):
    print(file)

Moved files to: /content/study2_results
failed_baseline_anthropic__claude-sonnet-4.5.json
failed_truth_labeled_anthropic__claude-sonnet-4.5.json
questions_truth_labeled_anthropic__claude-sonnet-4.5.json
questions_baseline_anthropic__claude-sonnet-4.5.json
fullgraph_normal_anthropic__claude-sonnet-4.5_8aae6891f8d1.json
failed_fullgraph_normal_anthropic__claude-sonnet-4.5_8aae6891f8d1.json
failed_fullgraph_normal_anthropic__claude-sonnet-4.5_b4fe236e516c.json
failed_truth_plain_anthropic__claude-sonnet-4.5.json
questions_truth_plain_anthropic__claude-sonnet-4.5.json
fullgraph_normal_anthropic__claude-sonnet-4.5_b4fe236e516c.json


In [ ]:
models = ['openai/gpt-4o',
          'openai/gpt-4o-mini',
          'anthropic/claude-sonnet-4.5',
          'meta-llama/llama-3.3-70b-instruct',
          'meta-llama/llama-3.1-8b-instruct']

In [ ]:
!rm -rf /content/study2_results*


In [ ]:
first_half = dataset['contexts'][:25]
manual = dataset['contexts'][25]
second_half = dataset['contexts'][26:]

In [ ]:
first_dataset = {
    **dataset,
    "contexts": first_half
}
second_dataset = {
    **dataset,
    "contexts": second_half
}

In [ ]:
ctx = dataset["contexts"][25]
_, dagType, _, passage, questions, _ = unpackDatasetItem(ctx)
graph = {
    "context": passage,
    "edges": [
        {
            "cause": "The Bundibugyo outbreak with no existing treatments",
            "effect": "Remdesivir being repurposed and tested"
        },
        {
            "cause": "The Bundibugyo outbreak with no existing treatments",
            "effect": "MBP-134 being identified as a potential treatment"
        },
        {
            "cause": "MBP-134 being identified as a potential treatment",
            "effect": "BARDA funding and donating MBP-134 doses for clinical trial"
        }
    ]
}
g = fullGraph()
for qKey, q in questions.items():
    opts = list(q["options"].values())
    plain = g.answer(
        "anthropic/claude-sonnet-4.5",
        graph,
        opts,
        passage,
        question=q["question"],
        labelThirdVars=False,
        returnRaw=True
    )
    labeled = g.answer(
        "anthropic/claude-sonnet-4.5",
        graph,
        opts,
        passage,
        question=q["question"],
        labelThirdVars=True,
        returnRaw=True
    )
    print(qKey, "correct:", q["answer"], "| plain:", plain, "| labeled:", labeled)

Q1 correct: A | plain: (1, '1') | labeled: (1, '1')
Q2 correct: D | plain: (3, '3') | labeled: (3, '3')
Q3 correct: D | plain: (2, '2') | labeled: (2, '2')
Q4 correct: D | plain: (3, '3') | labeled: (3, '3')
Q5 correct: D | plain: (1, '1') | labeled: (1, '1')


In [ ]:
r = runFinalEval(
    'anthropic/claude-sonnet-4.5',
    dataset,
    trueDAGs,
    runQuickCondition=True
)


QUICKGRAPH
chain_01 complete | Graphs: 1/32 | Question records: 10/320
chain_02 complete | Graphs: 2/32 | Question records: 20/320
chain_03 complete | Graphs: 3/32 | Question records: 30/320
chain_04 complete | Graphs: 4/32 | Question records: 40/320
fork_01 complete | Graphs: 5/32 | Question records: 50/320
fork_02 complete | Graphs: 6/32 | Question records: 60/320
fork_03 complete | Graphs: 7/32 | Question records: 70/320
fork_04 complete | Graphs: 8/32 | Question records: 80/320
collider_01 complete | Graphs: 9/32 | Question records: 90/320
collider_02 complete | Graphs: 10/32 | Question records: 100/320
collider_03 complete | Graphs: 11/32 | Question records: 110/320
collider_04 complete | Graphs: 12/32 | Question records: 120/320
diamond_01 complete | Graphs: 13/32 | Question records: 130/320
diamond_02 complete | Graphs: 14/32 | Question records: 140/320
diamond_03 complete | Graphs: 15/32 | Question records: 150/320
diamond_04 complete | Graphs: 16/32 | Question records: 160/32

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, json
ROOT = "/content/drive/MyDrive/models"
for root, dirs, files in os.walk(ROOT):
    level = root.replace(ROOT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    for file in sorted(files):
        path = os.path.join(root, file)
        print(f"{indent}  📄 {file}")
        if file.endswith(".json"):
            try:
                with open(path) as f:
                    data = json.load(f)
                if isinstance(data, dict):
                    print(f"{indent}     keys: {list(data.keys())}")
                    if "questionResults" in data:
                        print(f"{indent}     questionResults: {len(data['questionResults'])}")
                    if "generatedGraphs" in data:
                        print(f"{indent}     generatedGraphs: {len(data['generatedGraphs'])}")
                elif isinstance(data, list):
                    print(f"{indent}     list length: {len(data)}")
            except Exception as e:
                print(f"{indent}     ERROR: {e}")

Mounted at /content/drive
📁 models/
  📁 claude/
    📄 fullgraph_normal_anthropic__claude-sonnet-4.5_MERGED_REPAIRED.json
       keys: ['model', 'promptMediator', 'datasetFingerprint', 'contextIndex', 'questionIndex', 'stage', 'currentContextId', 'currentGraph', 'accuracyReportPlain', 'accuracyReportLabeled', 'totalQuestionsPlain', 'correctQuestionsPlain', 'totalAttemptedPlain', 'totalQuestionsLabeled', 'correctQuestionsLabeled', 'totalAttemptedLabeled', 'generatedGraphs', 'questionResults', 'invalidResponses', 'apiErrors', 'complete', 'totalAccuracyPlain', 'totalAccuracyLabeled', 'dagAccuracyPlain', 'dagAccuracyLabeled', 'repairMetadata']
       questionResults: 320
       generatedGraphs: 32
    📄 graphs_quick_anthropic__claude-sonnet-4.5.json
       keys: ['model', 'generatedGraphs', 'graphErrors']
       generatedGraphs: 32
    📄 questions_baseline_anthropic__claude-sonnet-4.5.json
       list length: 160
    📄 questions_quick_anthropic__claude-sonnet-4.5.json
       list length: 32

In [ ]:
import json
path = "/content/drive/MyDrive/models/true_dags.json"
with open(path) as f:
    true_dags = json.load(f)
print(type(true_dags))
print("length:", len(true_dags))
if isinstance(true_dags, dict):
    print("keys:", true_dags.keys())
    first = next(iter(true_dags.values()))
else:
    first = true_dags[0]
print("\nFIRST ITEM:")
print(first)

<class 'list'>
length: 32

FIRST ITEM:
{'contextId': 'chain_01', 'dag': [{'cause': 'Euclid surveying the distant universe', 'effect': '31 quasars being discovered'}, {'cause': '31 quasars being discovered', 'effect': '2 ancient quasars being identified using redlight technology'}, {'cause': '2 ancient quasars being identified using redlight technology', 'effect': 'Understanding of early black hole growth advancing'}]}


In [ ]:
from pathlib import Path
import os
import json
import zlib
import numpy as np
import pandas as pd
from scipy.stats import binomtest, wilcoxon
ROOT = Path("/content/drive/MyDrive/models")
TRUE_DAG_PATH = ROOT / "true_dags.json"
OUT = ROOT / "analysis_outputs"
OUT.mkdir(exist_ok=True)
BOOT_ITERS = 10_000
MODEL_NAMES = {
    "openai/gpt-4o":
        "GPT-4o",
    "openai/gpt-4o-mini":
        "GPT-4o-mini",
    "anthropic/claude-sonnet-4.5":
        "Claude Sonnet 4.5",
    "meta-llama/llama-3.3-70b-instruct":
        "Llama 3.3 70B",
    "meta-llama/llama-3.1-8b-instruct":
        "Llama 3.1 8B",
}
PROVIDER_MAP = {
    "openai/gpt-4o":
        "OpenAI",
    "openai/gpt-4o-mini":
        "OpenAI",
    "anthropic/claude-sonnet-4.5":
        "Amazon Bedrock",
    "meta-llama/llama-3.3-70b-instruct":
        "DeepInfra",
    "meta-llama/llama-3.1-8b-instruct":
        "DeepInfra",
}
CONDITIONS = [
    "baseline",
    "truth_plain",
    "truth_labeled",
    "full_plain",
    "full_labeled",
    "quick_plain",
    "quick_labeled",
]
PRIMARY_COMPARISONS = [
    ("baseline", "truth_plain"),
    ("baseline", "truth_labeled"),
    ("baseline", "full_plain"),
    ("baseline", "full_labeled"),
    ("baseline", "quick_plain"),
    ("baseline", "quick_labeled"),
    ("full_plain", "quick_plain"),
    ("full_labeled", "quick_labeled"),
]
ANNOTATION_COMPARISONS = [
    ("truth_plain", "truth_labeled"),
    ("full_plain", "full_labeled"),
    ("quick_plain", "quick_labeled"),
]

def read_json(path):
    with open(path, "r") as f:
        return json.load(f)

def only(folder, pattern):
    hits = sorted(folder.glob(pattern))
    if len(hits) != 1:
        raise RuntimeError(
            f"{folder.name}: expected exactly one "
            f"{pattern}, found {len(hits)}\n"
            f"{[x.name for x in hits]}"
        )
    return hits[0]

def stable_seed(*parts):
    text = "|".join(
        map(str, parts)
    ).encode("utf-8")
    return (
        zlib.crc32(text)
        & 0xffffffff
    )

def bootstrap_mean_ci(
    values,
    iters=BOOT_ITERS,
    seed=42
):
    values = np.asarray(
        values,
        dtype=float
    )
    values = values[
        np.isfinite(values)
    ]
    if len(values) == 0:
        return np.nan, np.nan
    if len(values) == 1:
        return (
            float(values[0]),
            float(values[0])
        )
    rng = np.random.default_rng(seed)
    samples = rng.integers(
        0,
        len(values),
        size=(
            iters,
            len(values)
        )
    )
    boot = values[
        samples
    ].mean(axis=1)
    low, high = np.quantile(
        boot,
        [0.025, 0.975]
    )
    return (
        float(low),
        float(high)
    )

def exact_mcnemar_p(
    a_only,
    b_only
):
    discordant = (
        int(a_only)
        + int(b_only)
    )
    if discordant == 0:
        return 1.0
    return float(
        binomtest(
            int(b_only),
            discordant,
            0.5,
            alternative="two-sided"
        ).pvalue
    )

def safe_wilcoxon(values):
    values = np.asarray(
        values,
        dtype=float
    )
    values = values[
        np.isfinite(values)
    ]
    if (
        len(values) == 0
        or np.allclose(values, 0)
    ):
        return 1.0
    try:
        return float(
            wilcoxon(
                values,
                zero_method="wilcox",
                alternative="two-sided",
                method="auto"
            ).pvalue
        )
    except ValueError:
        return 1.0

def holm_adjust(p_values):
    p = np.asarray(
        p_values,
        dtype=float
    )
    result = np.full(
        len(p),
        np.nan
    )
    valid_mask = np.isfinite(p)
    valid_indices = np.where(
        valid_mask
    )[0]
    valid_p = p[
        valid_mask
    ]
    if len(valid_p) == 0:
        return result
    order = np.argsort(
        valid_p
    )
    ordered = valid_p[
        order
    ]
    m = len(ordered)
    adjusted_ordered = (
        np.empty(
            m,
            dtype=float
        )
    )
    running = 0.0
    for i, raw_p in enumerate(
        ordered
    ):
        adjusted = (
            (m - i)
            * raw_p
        )
        running = max(
            running,
            adjusted
        )
        adjusted_ordered[i] = min(
            running,
            1.0
        )
    reverse = np.empty(
        m,
        dtype=int
    )
    reverse[
        order
    ] = np.arange(m)
    result[
        valid_indices
    ] = adjusted_ordered[
        reverse
    ]
    return result

def normalize_edge(edge):
    return (
        str(
            edge["cause"]
        ).strip(),
        str(
            edge["effect"]
        ).strip()
    )

def edges_to_set(edges):
    return {
        normalize_edge(edge)
        for edge in edges
    }
true_dags = read_json(
    TRUE_DAG_PATH
)
assert isinstance(
    true_dags,
    list
)
assert len(
    true_dags
) == 32
truth_map = {
    item["contextId"]:
        item["dag"]
    for item in true_dags
}
assert len(
    truth_map
) == 32
truth_context_ids = set(
    truth_map.keys()
)
print(
    "True DAGs loaded:",
    len(truth_map)
)
model_folders = sorted([
    folder
    for folder in ROOT.iterdir()
    if (
        folder.is_dir()
        and
        folder.name
        != "analysis_outputs"
        and
        list(
            folder.glob(
                "questions_baseline_*.json"
            )
        )
    )
])
print(
    "Model folders:",
    [
        x.name
        for x in model_folders
    ]
)
question_rows = []
graph_rows = []
manifest_rows = []
for folder in model_folders:
    files = {
        "baseline":
            only(
                folder,
                "questions_baseline_*.json"
            ),
        "truth_plain":
            only(
                folder,
                "questions_truth_plain_*.json"
            ),
        "truth_labeled":
            only(
                folder,
                "questions_truth_labeled_*.json"
            ),
        "full":
            only(
                folder,
                "fullgraph_normal_*.json"
            ),
        "quick_questions":
            only(
                folder,
                "questions_quick_*.json"
            ),
        "quick_graphs":
            only(
                folder,
                "graphs_quick_*.json"
            ),
    }
    baseline = read_json(
        files["baseline"]
    )
    truth_plain = read_json(
        files["truth_plain"]
    )
    truth_labeled = read_json(
        files["truth_labeled"]
    )
    full = read_json(
        files["full"]
    )
    quick_questions = read_json(
        files["quick_questions"]
    )
    quick_graphs = read_json(
        files["quick_graphs"]
    )
    model = (
        full.get("model")
        or
        quick_graphs.get("model")
        or
        baseline[0].get("model")
    )
    analysis_model = MODEL_NAMES.get(
        model,
        model
    )
    source_packs = [
        (
            "baseline",
            baseline,
            files["baseline"]
        ),
        (
            "truth_plain",
            truth_plain,
            files["truth_plain"]
        ),
        (
            "truth_labeled",
            truth_labeled,
            files["truth_labeled"]
        ),
    ]
    for (
        condition,
        rows,
        source_file
    ) in source_packs:
        for row in rows:
            r = dict(row)
            r["condition"] = condition
            r[
                "analysis_model"
            ] = analysis_model
            r[
                "model_id"
            ] = model
            r[
                "model_folder"
            ] = folder.name
            r[
                "source_file"
            ] = source_file.name
            question_rows.append(
                r
            )
    for row in full[
        "questionResults"
    ]:
        r = dict(row)
        r[
            "analysis_model"
        ] = analysis_model
        r[
            "model_id"
        ] = model
        r[
            "model_folder"
        ] = folder.name
        r[
            "source_file"
        ] = files[
            "full"
        ].name
        question_rows.append(
            r
        )
    for row in quick_questions:
        r = dict(row)
        r[
            "analysis_model"
        ] = analysis_model
        r[
            "model_id"
        ] = model
        r[
            "model_folder"
        ] = folder.name
        r[
            "source_file"
        ] = files[
            "quick_questions"
        ].name
        question_rows.append(
            r
        )
    graph_sources = [
        (
            "full",
            full[
                "generatedGraphs"
            ],
            files["full"]
        ),
        (
            "quick",
            quick_graphs[
                "generatedGraphs"
            ],
            files[
                "quick_graphs"
            ]
        ),
    ]
    for (
        method,
        graphs,
        source_file
    ) in graph_sources:
        for graph in graphs:
            graph_rows.append({
                "analysis_model":
                    analysis_model,
                "model_id":
                    model,
                "model_folder":
                    folder.name,
                "graph_method":
                    method,
                "context_id":
                    graph[
                        "contextId"
                    ],
                "pred_edges":
                    graph[
                        "graph"
                    ],
                "source_file":
                    source_file.name
            })
    manifest_rows.append({
        "analysis_model":
            analysis_model,
        "model_id":
            model,
        "folder":
            folder.name,
        "baseline_file":
            files[
                "baseline"
            ].name,
        "truth_plain_file":
            files[
                "truth_plain"
            ].name,
        "truth_labeled_file":
            files[
                "truth_labeled"
            ].name,
        "full_file":
            files[
                "full"
            ].name,
        "quick_questions_file":
            files[
                "quick_questions"
            ].name,
        "quick_graph_file":
            files[
                "quick_graphs"
            ].name,
        "full_file_apiErrors":
            len(
                full.get(
                    "apiErrors",
                    []
                )
            ),
        "full_file_invalidResponses":
            len(
                full.get(
                    "invalidResponses",
                    []
                )
            ),
        "quick_graphErrors":
            len(
                quick_graphs.get(
                    "graphErrors",
                    []
                )
            )
    })
question_df = pd.DataFrame(
    question_rows
)
graph_raw_df = pd.DataFrame(
    graph_rows
)
manifest_df = pd.DataFrame(
    manifest_rows
)
manifest_df.to_csv(
    OUT
    / "00_source_manifest.csv",
    index=False
)
print(
    "\nQuestion rows loaded:",
    len(question_df)
)
print(
    "Graph rows loaded:",
    len(graph_raw_df)
)
qa_rows = []
critical_errors = []
for (
    model_name,
    model_df
) in question_df.groupby(
    "analysis_model"
):
    baseline_ids = set(
        model_df.loc[
            model_df[
                "condition"
            ] == "baseline",
            "question_id"
        ].astype(str)
    )
    for condition in CONDITIONS:
        condition_df = model_df[
            model_df[
                "condition"
            ] == condition
        ].copy()
        ids = condition_df[
            "question_id"
        ].astype(str)
        id_set = set(ids)
        duplicates = int(
            ids.duplicated().sum()
        )
        missing = len(
            baseline_ids
            - id_set
        )
        extra = len(
            id_set
            - baseline_ids
        )
        null_correct = int(
            condition_df[
                "correct"
            ].isna().sum()
        )
        invalid = int(
            (
                condition_df[
                    "status"
                ] == "invalid"
            ).sum()
        )
        refusals = int(
            (
                condition_df[
                    "status"
                ] == "refusal"
            ).sum()
        )
        api_errors = int(
            (
                condition_df[
                    "status"
                ] == "api_error"
            ).sum()
        )
        context_sizes = (
            condition_df
            .groupby(
                "context_id"
            )
            .size()
        )
        contexts_correct = (
            len(
                context_sizes
            ) == 32
            and
            set(
                context_sizes.values
            ) == {5}
        )
        passed = (
            len(
                condition_df
            ) == 160
            and
            len(
                id_set
            ) == 160
            and
            duplicates == 0
            and
            missing == 0
            and
            extra == 0
            and
            null_correct == 0
            and
            api_errors == 0
            and
            contexts_correct
        )
        qa_rows.append({
            "audit_type":
                "question_condition",
            "analysis_model":
                model_name,
            "condition_or_method":
                condition,
            "rows":
                len(
                    condition_df
                ),
            "unique_ids":
                len(
                    id_set
                ),
            "contexts":
                len(
                    context_sizes
                ),
            "all_contexts_have_5":
                contexts_correct,
            "duplicates":
                duplicates,
            "missing_vs_baseline":
                missing,
            "extra_vs_baseline":
                extra,
            "null_correct":
                null_correct,
            "invalid":
                invalid,
            "refusals":
                refusals,
            "api_errors":
                api_errors,
            "passed":
                passed
        })
        if not passed:
            critical_errors.append(
                f"{model_name} / "
                f"{condition}"
            )
for (
    model_name,
    method
), graph_df in graph_raw_df.groupby(
    [
        "analysis_model",
        "graph_method"
    ]
):
    ids = graph_df[
        "context_id"
    ].astype(str)
    id_set = set(ids)
    duplicates = int(
        ids.duplicated().sum()
    )
    missing = len(
        truth_context_ids
        - id_set
    )
    extra = len(
        id_set
        - truth_context_ids
    )
    passed = (
        len(
            graph_df
        ) == 32
        and
        len(
            id_set
        ) == 32
        and
        duplicates == 0
        and
        missing == 0
        and
        extra == 0
    )
    qa_rows.append({
        "audit_type":
            "graph_method",
        "analysis_model":
            model_name,
        "condition_or_method":
            method,
        "rows":
            len(
                graph_df
            ),
        "unique_ids":
            len(
                id_set
            ),
        "contexts":
            len(
                id_set
            ),
        "all_contexts_have_5":
            np.nan,
        "duplicates":
            duplicates,
        "missing_vs_baseline":
            missing,
        "extra_vs_baseline":
            extra,
        "null_correct":
            np.nan,
        "invalid":
            np.nan,
        "refusals":
            np.nan,
        "api_errors":
            np.nan,
        "passed":
            passed
    })
    if not passed:
        critical_errors.append(
            f"{model_name} / "
            f"{method} graphs"
        )
qa_df = pd.DataFrame(
    qa_rows
)
qa_df.to_csv(
    OUT
    / "09_quality_audit.csv",
    index=False
)
if critical_errors:
    print(
        qa_df.to_string(
            index=False
        )
    )
    raise RuntimeError(
        "\nQA FAILED. "
        "Fix these before statistics:\n"
        +
        "\n".join(
            critical_errors
        )
    )
print(
    "\nQA PASSED"
)
available_models = set(
    question_df[
        "analysis_model"
    ]
)
model_order = [
    name
    for name in MODEL_NAMES.values()
    if name in available_models
]
accuracy_rows = []
for (
    model_name,
    condition
), condition_df in question_df.groupby(
    [
        "analysis_model",
        "condition"
    ]
):
    condition_df = (
        condition_df.copy()
    )
    condition_df[
        "correct"
    ] = condition_df[
        "correct"
    ].astype(float)
    context_accuracy = (
        condition_df
        .groupby(
            "context_id"
        )[
            "correct"
        ]
        .mean()
        .sort_index()
    )
    low, high = (
        bootstrap_mean_ci(
            context_accuracy.values,
            seed=stable_seed(
                model_name,
                condition,
                "accuracy"
            )
        )
    )
    correct_count = int(
        condition_df[
            "correct"
        ].sum()
    )
    N = len(
        condition_df
    )
    accuracy = (
        correct_count
        / N
    )
    accuracy_rows.append({
        "analysis_model":
            model_name,
        "condition":
            condition,
        "correct":
            correct_count,
        "N":
            N,
        "accuracy":
            accuracy,
        "accuracy_pct":
            100 * accuracy,
        "ci95_low":
            low,
        "ci95_high":
            high,
        "ci95_low_pct":
            100 * low,
        "ci95_high_pct":
            100 * high,
        "display":
            (
                f"{correct_count}/{N} "
                f"({100*accuracy:.2f}%)"
            )
    })
accuracy_df = pd.DataFrame(
    accuracy_rows
)
condition_order = {
    condition: i
    for i, condition
    in enumerate(
        CONDITIONS
    )
}
accuracy_df[
    "condition_order"
] = accuracy_df[
    "condition"
].map(
    condition_order
)
accuracy_df = (
    accuracy_df
    .sort_values(
        [
            "analysis_model",
            "condition_order"
        ]
    )
    .drop(
        columns=[
            "condition_order"
        ]
    )
)
accuracy_df.to_csv(
    OUT
    / "01_accuracy_summary.csv",
    index=False
)

def compare_conditions(
    model_name,
    condition_a,
    condition_b
):
    a = question_df[
        (
            question_df[
                "analysis_model"
            ] == model_name
        )
        &
        (
            question_df[
                "condition"
            ] == condition_a
        )
    ][
        [
            "question_id",
            "context_id",
            "correct"
        ]
    ].copy()
    b = question_df[
        (
            question_df[
                "analysis_model"
            ] == model_name
        )
        &
        (
            question_df[
                "condition"
            ] == condition_b
        )
    ][
        [
            "question_id",
            "context_id",
            "correct"
        ]
    ].copy()
    if (
        set(
            a[
                "question_id"
            ]
        )
        !=
        set(
            b[
                "question_id"
            ]
        )
    ):
        raise RuntimeError(
            f"Pair mismatch: "
            f"{model_name} | "
            f"{condition_a} vs "
            f"{condition_b}"
        )
    merged = a.merge(
        b,
        on=[
            "question_id",
            "context_id"
        ],
        suffixes=(
            "_a",
            "_b"
        ),
        validate="one_to_one"
    )
    if len(
        merged
    ) != 160:
        raise RuntimeError(
            f"Expected 160 pairs: "
            f"{model_name} | "
            f"{condition_a} vs "
            f"{condition_b}"
        )
    merged[
        "correct_a"
    ] = merged[
        "correct_a"
    ].astype(int)
    merged[
        "correct_b"
    ] = merged[
        "correct_b"
    ].astype(int)
    a_only = int(
        (
            (
                merged[
                    "correct_a"
                ] == 1
            )
            &
            (
                merged[
                    "correct_b"
                ] == 0
            )
        ).sum()
    )
    b_only = int(
        (
            (
                merged[
                    "correct_a"
                ] == 0
            )
            &
            (
                merged[
                    "correct_b"
                ] == 1
            )
        ).sum()
    )
    accuracy_a = float(
        merged[
            "correct_a"
        ].mean()
    )
    accuracy_b = float(
        merged[
            "correct_b"
        ].mean()
    )
    delta = (
        accuracy_b
        - accuracy_a
    )
    context_delta = (
        merged
        .assign(
            difference=(
                merged[
                    "correct_b"
                ]
                -
                merged[
                    "correct_a"
                ]
            )
        )
        .groupby(
            "context_id"
        )[
            "difference"
        ]
        .mean()
        .sort_index()
    )
    low, high = (
        bootstrap_mean_ci(
            context_delta.values,
            seed=stable_seed(
                model_name,
                condition_a,
                condition_b,
                "delta"
            )
        )
    )
    return {
        "analysis_model":
            model_name,
        "condition_a":
            condition_a,
        "condition_b":
            condition_b,
        "comparison":
            (
                f"{condition_b} - "
                f"{condition_a}"
            ),
        "N_pairs":
            len(
                merged
            ),
        "n_contexts":
            len(
                context_delta
            ),
        "a_correct_b_wrong":
            a_only,
        "a_wrong_b_correct":
            b_only,
        "discordant":
            (
                a_only
                + b_only
            ),
        "accuracy_a":
            accuracy_a,
        "accuracy_b":
            accuracy_b,
        "delta":
            delta,
        "delta_pp":
            100 * delta,
        "ci95_low":
            low,
        "ci95_high":
            high,
        "ci95_low_pp":
            100 * low,
        "ci95_high_pp":
            100 * high,
        "mcnemar_p":
            exact_mcnemar_p(
                a_only,
                b_only
            ),
        "context_wilcoxon_p":
            safe_wilcoxon(
                context_delta.values
            ),
        "paired_ids_match":
            True
    }
downstream_rows = []
for model_name in model_order:
    for (
        condition_a,
        condition_b
    ) in PRIMARY_COMPARISONS:
        downstream_rows.append(
            compare_conditions(
                model_name,
                condition_a,
                condition_b
            )
        )
downstream_df = pd.DataFrame(
    downstream_rows
)
downstream_df[
    "mcnemar_p_holm_by_contrast"
] = np.nan
downstream_df[
    "context_wilcoxon_p_holm_by_contrast"
] = np.nan
for (
    comparison,
    indexes
) in downstream_df.groupby(
    "comparison"
).groups.items():
    indexes = list(
        indexes
    )
    downstream_df.loc[
        indexes,
        "mcnemar_p_holm_by_contrast"
    ] = holm_adjust(
        downstream_df.loc[
            indexes,
            "mcnemar_p"
        ].values
    )
    downstream_df.loc[
        indexes,
        "context_wilcoxon_p_holm_by_contrast"
    ] = holm_adjust(
        downstream_df.loc[
            indexes,
            "context_wilcoxon_p"
        ].values
    )
downstream_df[
    "mcnemar_p_holm_global"
] = holm_adjust(
    downstream_df[
        "mcnemar_p"
    ].values
)
downstream_df.to_csv(
    OUT
    / "02_downstream_paired_tests.csv",
    index=False
)
annotation_rows = []
for model_name in model_order:
    for (
        condition_a,
        condition_b
    ) in ANNOTATION_COMPARISONS:
        result = compare_conditions(
            model_name,
            condition_a,
            condition_b
        )
        result[
            "representation"
        ] = condition_a.replace(
            "_plain",
            ""
        )
        annotation_rows.append(
            result
        )
annotation_df = pd.DataFrame(
    annotation_rows
)
annotation_df[
    "mcnemar_p_holm_by_representation"
] = np.nan
for (
    representation,
    indexes
) in annotation_df.groupby(
    "representation"
).groups.items():
    indexes = list(
        indexes
    )
    annotation_df.loc[
        indexes,
        "mcnemar_p_holm_by_representation"
    ] = holm_adjust(
        annotation_df.loc[
            indexes,
            "mcnemar_p"
        ].values
    )
annotation_df[
    "mcnemar_p_holm_global"
] = holm_adjust(
    annotation_df[
        "mcnemar_p"
    ].values
)
annotation_df.to_csv(
    OUT
    / "03_annotation_effects.csv",
    index=False
)
graph_metric_rows = []
for _, row in graph_raw_df.iterrows():
    context_id = row[
        "context_id"
    ]
    predicted = edges_to_set(
        row[
            "pred_edges"
        ]
    )
    truth = edges_to_set(
        truth_map[
            context_id
        ]
    )
    TP = len(
        predicted
        & truth
    )
    FP = len(
        predicted
        - truth
    )
    FN = len(
        truth
        - predicted
    )
    precision = (
        TP
        / (TP + FP)
        if (
            TP + FP
        )
        else 0.0
    )
    recall = (
        TP
        / (TP + FN)
        if (
            TP + FN
        )
        else 0.0
    )
    if (
        precision
        + recall
    ):
        f1 = (
            2
            * precision
            * recall
            /
            (
                precision
                + recall
            )
        )
    else:
        f1 = 0.0
    exact = int(
        predicted
        == truth
    )
    edit_distance = (
        FP
        + FN
    )
    graph_metric_rows.append({
        "analysis_model":
            row[
                "analysis_model"
            ],
        "model_id":
            row[
                "model_id"
            ],
        "graph_method":
            row[
                "graph_method"
            ],
        "context_id":
            context_id,
        "true_edge_count":
            len(
                truth
            ),
        "pred_edge_count":
            len(
                predicted
            ),
        "TP":
            TP,
        "FP":
            FP,
        "FN":
            FN,
        "precision":
            precision,
        "recall":
            recall,
        "f1":
            f1,
        "edit_distance":
            edit_distance,
        "exact_match":
            exact,
        "pred_edges_json":
            json.dumps(
                row[
                    "pred_edges"
                ],
                ensure_ascii=False
            ),
        "true_edges_json":
            json.dumps(
                truth_map[
                    context_id
                ],
                ensure_ascii=False
            ),
        "source_file":
            row[
                "source_file"
            ]
    })
graph_context_df = pd.DataFrame(
    graph_metric_rows
)
graph_context_df.to_csv(
    OUT
    / "04_graph_metrics_context.csv",
    index=False
)
graph_summary_rows = []
for (
    model_name,
    method
), graph_df in graph_context_df.groupby(
    [
        "analysis_model",
        "graph_method"
    ]
):
    TP = int(
        graph_df[
            "TP"
        ].sum()
    )
    FP = int(
        graph_df[
            "FP"
        ].sum()
    )
    FN = int(
        graph_df[
            "FN"
        ].sum()
    )
    micro_precision = (
        TP
        / (TP + FP)
        if (
            TP + FP
        )
        else 0.0
    )
    micro_recall = (
        TP
        / (TP + FN)
        if (
            TP + FN
        )
        else 0.0
    )
    micro_f1 = (
        (
            2
            * micro_precision
            * micro_recall
            /
            (
                micro_precision
                + micro_recall
            )
        )
        if (
            micro_precision
            + micro_recall
        )
        else 0.0
    )
    f1_low, f1_high = (
        bootstrap_mean_ci(
            graph_df[
                "f1"
            ].values,
            seed=stable_seed(
                model_name,
                method,
                "graph_f1"
            )
        )
    )
    edit_low, edit_high = (
        bootstrap_mean_ci(
            graph_df[
                "edit_distance"
            ].values,
            seed=stable_seed(
                model_name,
                method,
                "graph_edit"
            )
        )
    )
    exact_low, exact_high = (
        bootstrap_mean_ci(
            graph_df[
                "exact_match"
            ].values,
            seed=stable_seed(
                model_name,
                method,
                "graph_exact"
            )
        )
    )
    graph_summary_rows.append({
        "analysis_model":
            model_name,
        "graph_method":
            method,
        "contexts":
            len(
                graph_df
            ),
        "TP":
            TP,
        "FP":
            FP,
        "FN":
            FN,
        "micro_precision":
            micro_precision,
        "micro_recall":
            micro_recall,
        "micro_f1":
            micro_f1,
        "mean_context_f1":
            float(
                graph_df[
                    "f1"
                ].mean()
            ),
        "median_context_f1":
            float(
                graph_df[
                    "f1"
                ].median()
            ),
        "mean_context_f1_ci95_low":
            f1_low,
        "mean_context_f1_ci95_high":
            f1_high,
        "mean_edit_distance":
            float(
                graph_df[
                    "edit_distance"
                ].mean()
            ),
        "mean_edit_distance_ci95_low":
            edit_low,
        "mean_edit_distance_ci95_high":
            edit_high,
        "exact_matches":
            int(
                graph_df[
                    "exact_match"
                ].sum()
            ),
        "exact_rate":
            float(
                graph_df[
                    "exact_match"
                ].mean()
            ),
        "exact_rate_ci95_low":
            exact_low,
        "exact_rate_ci95_high":
            exact_high
    })
graph_summary_df = pd.DataFrame(
    graph_summary_rows
)
graph_summary_df.to_csv(
    OUT
    / "05_graph_summary.csv",
    index=False
)
graph_test_rows = []
for model_name in model_order:
    full = graph_context_df[
        (
            graph_context_df[
                "analysis_model"
            ] == model_name
        )
        &
        (
            graph_context_df[
                "graph_method"
            ] == "full"
        )
    ][
        [
            "context_id",
            "f1",
            "exact_match"
        ]
    ].copy()
    quick = graph_context_df[
        (
            graph_context_df[
                "analysis_model"
            ] == model_name
        )
        &
        (
            graph_context_df[
                "graph_method"
            ] == "quick"
        )
    ][
        [
            "context_id",
            "f1",
            "exact_match"
        ]
    ].copy()
    if (
        set(
            full[
                "context_id"
            ]
        )
        !=
        set(
            quick[
                "context_id"
            ]
        )
    ):
        raise RuntimeError(
            f"Graph pairing mismatch: "
            f"{model_name}"
        )
    paired = full.merge(
        quick,
        on="context_id",
        suffixes=(
            "_full",
            "_quick"
        ),
        validate="one_to_one"
    )
    paired[
        "delta_f1"
    ] = (
        paired[
            "f1_quick"
        ]
        -
        paired[
            "f1_full"
        ]
    )
    low, high = (
        bootstrap_mean_ci(
            paired[
                "delta_f1"
            ].values,
            seed=stable_seed(
                model_name,
                "graph_delta"
            )
        )
    )
    full_only = int(
        (
            (
                paired[
                    "exact_match_full"
                ] == 1
            )
            &
            (
                paired[
                    "exact_match_quick"
                ] == 0
            )
        ).sum()
    )
    quick_only = int(
        (
            (
                paired[
                    "exact_match_full"
                ] == 0
            )
            &
            (
                paired[
                    "exact_match_quick"
                ] == 1
            )
        ).sum()
    )
    full_micro_f1 = float(
        graph_summary_df[
            (
                graph_summary_df[
                    "analysis_model"
                ] == model_name
            )
            &
            (
                graph_summary_df[
                    "graph_method"
                ] == "full"
            )
        ][
            "micro_f1"
        ].iloc[0]
    )
    quick_micro_f1 = float(
        graph_summary_df[
            (
                graph_summary_df[
                    "analysis_model"
                ] == model_name
            )
            &
            (
                graph_summary_df[
                    "graph_method"
                ] == "quick"
            )
        ][
            "micro_f1"
        ].iloc[0]
    )
    graph_test_rows.append({
        "analysis_model":
            model_name,
        "contexts":
            32,
        "full_mean_context_f1":
            float(
                paired[
                    "f1_full"
                ].mean()
            ),
        "quick_mean_context_f1":
            float(
                paired[
                    "f1_quick"
                ].mean()
            ),
        "mean_delta_f1_quick_minus_full":
            float(
                paired[
                    "delta_f1"
                ].mean()
            ),
        "median_delta_f1_quick_minus_full":
            float(
                paired[
                    "delta_f1"
                ].median()
            ),
        "delta_f1_ci95_low":
            low,
        "delta_f1_ci95_high":
            high,
        "wilcoxon_p":
            safe_wilcoxon(
                paired[
                    "delta_f1"
                ].values
            ),
        "full_micro_f1":
            full_micro_f1,
        "quick_micro_f1":
            quick_micro_f1,
        "micro_f1_delta_quick_minus_full":
            (
                quick_micro_f1
                -
                full_micro_f1
            ),
        "full_exact_quick_not":
            full_only,
        "quick_exact_full_not":
            quick_only,
        "exact_discordant":
            (
                full_only
                + quick_only
            ),
        "exact_mcnemar_p":
            exact_mcnemar_p(
                full_only,
                quick_only
            )
    })
graph_tests_df = pd.DataFrame(
    graph_test_rows
)
graph_tests_df[
    "wilcoxon_p_holm"
] = holm_adjust(
    graph_tests_df[
        "wilcoxon_p"
    ].values
)
graph_tests_df[
    "exact_mcnemar_p_holm"
] = holm_adjust(
    graph_tests_df[
        "exact_mcnemar_p"
    ].values
)
graph_tests_df.to_csv(
    OUT
    / "06_full_vs_quick_fidelity_tests.csv",
    index=False
)
fidelity_utility_rows = []
for model_name in model_order:
    graph_result = graph_tests_df[
        graph_tests_df[
            "analysis_model"
        ] == model_name
    ].iloc[0]
    plain_result = downstream_df[
        (
            downstream_df[
                "analysis_model"
            ] == model_name
        )
        &
        (
            downstream_df[
                "condition_a"
            ] == "full_plain"
        )
        &
        (
            downstream_df[
                "condition_b"
            ] == "quick_plain"
        )
    ].iloc[0]
    labeled_result = downstream_df[
        (
            downstream_df[
                "analysis_model"
            ] == model_name
        )
        &
        (
            downstream_df[
                "condition_a"
            ] == "full_labeled"
        )
        &
        (
            downstream_df[
                "condition_b"
            ] == "quick_labeled"
        )
    ].iloc[0]
    fidelity_utility_rows.append({
        "analysis_model":
            model_name,
        "graph_micro_f1_delta_quick_minus_full":
            graph_result[
                "micro_f1_delta_quick_minus_full"
            ],
        "graph_mean_context_f1_delta_quick_minus_full":
            graph_result[
                "mean_delta_f1_quick_minus_full"
            ],
        "graph_delta_f1_ci95_low":
            graph_result[
                "delta_f1_ci95_low"
            ],
        "graph_delta_f1_ci95_high":
            graph_result[
                "delta_f1_ci95_high"
            ],
        "graph_wilcoxon_p_holm":
            graph_result[
                "wilcoxon_p_holm"
            ],
        "plain_delta_pp":
            plain_result[
                "delta_pp"
            ],
        "plain_ci95_low_pp":
            plain_result[
                "ci95_low_pp"
            ],
        "plain_ci95_high_pp":
            plain_result[
                "ci95_high_pp"
            ],
        "plain_mcnemar_p_holm":
            plain_result[
                "mcnemar_p_holm_by_contrast"
            ],
        "labeled_delta_pp":
            labeled_result[
                "delta_pp"
            ],
        "labeled_ci95_low_pp":
            labeled_result[
                "ci95_low_pp"
            ],
        "labeled_ci95_high_pp":
            labeled_result[
                "ci95_high_pp"
            ],
        "labeled_mcnemar_p_holm":
            labeled_result[
                "mcnemar_p_holm_by_contrast"
            ]
    })
fidelity_utility_df = (
    pd.DataFrame(
        fidelity_utility_rows
    )
)
fidelity_utility_df.to_csv(
    OUT
    / "07_fidelity_vs_utility.csv",
    index=False
)
topology_df = (
    question_df
    .groupby(
        [
            "analysis_model",
            "condition",
            "topology"
        ],
        as_index=False
    )
    .agg(
        correct=(
            "correct",
            "sum"
        ),
        N=(
            "correct",
            "size"
        )
    )
)
topology_df[
    "correct"
] = topology_df[
    "correct"
].astype(int)
topology_df[
    "accuracy"
] = (
    topology_df[
        "correct"
    ]
    /
    topology_df[
        "N"
    ]
)
topology_df[
    "accuracy_pct"
] = (
    100
    * topology_df[
        "accuracy"
    ]
)
topology_df.to_csv(
    OUT
    / "08_topology_descriptive.csv",
    index=False
)
config_rows = []
for _, row in manifest_df.iterrows():
    model = row[
        "model_id"
    ]
    config_rows.append({
        "analysis_model":
            row[
                "analysis_model"
            ],
        "model_id":
            model,
        "routing_service":
            "OpenRouter",
        "requested_pinned_provider":
            PROVIDER_MAP.get(
                model,
                "UNKNOWN"
            ),
        "provider_fallback_disabled":
            True,
        "actual_provider_persisted_in_result_files":
            False,
        "temperature":
            0,
        "option_count":
            4,
        "chance_level":
            0.25,
        "contexts":
            32,
        "questions_per_context":
            5,
        "questions_per_condition":
            160,
        "baseline_truth_parser":
            "single A-D letter; otherwise invalid",
        "full_quick_parser":
            "single integer 1-4; otherwise invalid",
        "invalid_policy":
            "count as incorrect; no rerun",
        "api_error_policy":
            "must be repaired before final analysis",
        "refusal_policy":
            "audit separately; count incorrect if recorded as refusal"
    })
config_df = pd.DataFrame(
    config_rows
)
config_df.to_csv(
    OUT
    / "10_run_configuration.csv",
    index=False
)
question_df.to_csv(
    OUT
    / "11_question_level_master.csv",
    index=False
)
graph_context_df.to_csv(
    OUT
    / "12_graph_level_master.csv",
    index=False
)
accuracy_df.to_csv(
    OUT
    / "paper_table_1_accuracy.csv",
    index=False
)
downstream_df.to_csv(
    OUT
    / "paper_table_2_downstream_tests.csv",
    index=False
)
annotation_df.to_csv(
    OUT
    / "paper_table_3_annotation.csv",
    index=False
)
paper_graph = graph_summary_df.merge(
    graph_tests_df,
    on="analysis_model",
    how="left",
    suffixes=(
        "",
        "_comparison"
    )
)
paper_graph.to_csv(
    OUT
    / "paper_table_4_graph_fidelity.csv",
    index=False
)
fidelity_utility_df.to_csv(
    OUT
    / "paper_table_5_fidelity_utility.csv",
    index=False
)
print(
    "\n===================================="
)
print(
    "STUDY 2 ANALYSIS COMPLETE"
)
print(
    "===================================="
)
print(
    "Models:",
    len(
        model_order
    )
)
print(
    "Question rows:",
    len(
        question_df
    )
)
print(
    "Expected:",
    5 * 1120
)
print(
    "Graph rows:",
    len(
        graph_context_df
    )
)
print(
    "Expected:",
    5 * 64
)
print(
    "QA failures:",
    len(
        critical_errors
    )
)
print(
    "\nOutputs saved to:"
)
print(
    OUT
)
print(
    "\nACCURACY (%)"
)
print(
    accuracy_df
    .pivot(
        index="analysis_model",
        columns="condition",
        values="accuracy_pct"
    )
    .round(2)
    .to_string()
)
print(
    "\nFULL vs QUICK GRAPH FIDELITY"
)
print(
    graph_tests_df[
        [
            "analysis_model",
            "full_micro_f1",
            "quick_micro_f1",
            "mean_delta_f1_quick_minus_full",
            "wilcoxon_p_holm",
            "exact_mcnemar_p_holm"
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)

True DAGs loaded: 32
Model folders: ['claude', 'gpt4o', 'gpt4omini', 'llama3.18b', 'llama3.370b']

Question rows loaded: 5600
Graph rows loaded: 320

QA PASSED

STUDY 2 ANALYSIS COMPLETE
Models: 5
Question rows: 5600
Expected: 5600
Graph rows: 320
Expected: 320
QA failures: 0

Outputs saved to:
/content/drive/MyDrive/models/analysis_outputs

ACCURACY (%)
condition          baseline  full_labeled  full_plain  quick_labeled  quick_plain  truth_labeled  truth_plain
analysis_model                                                                                               
Claude Sonnet 4.5     36.25         33.75       29.38          41.25        35.62          42.50        41.25
GPT-4o                35.00         34.38       30.63          30.00        30.00          35.00        38.75
GPT-4o-mini           25.62         33.75       23.75          25.62        18.12          27.50        21.88
Llama 3.1 8B          31.25         20.62       21.25          20.00        17.50          21